# Global Information and Machine Learning in Indian Realized-Volatility Forecasting

In [2]:
import os, io, glob, fnmatch, warnings, time, hashlib
from collections import defaultdict
import numpy as np, pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from scipy.special import gamma as gamma_fn
warnings.filterwarnings("ignore")

RNG_SEED = 42
SEEDS = (0, 1, 2, 3, 4)
TRADING_DAYS_PER_YEAR = 252
SESSION_START, SESSION_END = "09:15", "15:30"
MIN_BARS_PER_DAY = 20
JUMP_ALPHA = 0.999
HORIZONS = [1, 5, 22]
PRIMARY_HORIZON = 1
OOS_FRACTION = 0.30
PRED_FLOOR_FRAC = 0.05
os.environ.setdefault("PYTHONWARNINGS", "ignore")

DATA_DIR_LOCAL = "Datasets"


# ---------------------------------------------------------------------
# INFORMATION-SET ALIGNMENT  (unchanged)
# ---------------------------------------------------------------------
def align_strictly_before(global_series, indian_dates, name):
    g = global_series.dropna().sort_index()
    left = pd.DataFrame({"ind_date": pd.DatetimeIndex(indian_dates)}).sort_values("ind_date")
    right = pd.DataFrame({"g_date": g.index, "val": g.values}).sort_values("g_date")
    m = pd.merge_asof(left, right, left_on="ind_date", right_on="g_date",
                      direction="backward", allow_exact_matches=False)
    m["lag_days"] = (m["ind_date"] - m["g_date"]).dt.days
    out = pd.Series(m["val"].values, index=pd.DatetimeIndex(m["ind_date"]), name=name)
    return out, m.set_index("ind_date")["lag_days"]


def assert_no_lookahead(global_series, indian_dates, aligned, name):
    g = global_series.dropna().sort_index()
    probe = pd.Timestamp(pd.DatetimeIndex(indian_dates)[int(0.6 * len(indian_dates))])
    g_bad = g.copy()
    g_bad[g_bad.index >= probe] = -12345.0
    a2, _ = align_strictly_before(g_bad, indian_dates, name)
    before = pd.DatetimeIndex(indian_dates) <= probe
    lhs = aligned[before].dropna()
    rhs = a2[before].reindex(lhs.index)
    assert np.allclose(lhs.values, rhs.values), \
        f"LOOKAHEAD in {name}: corrupting global data dated >= {probe.date()} " \
        f"changed an aligned value at or before that date"
    _, lag = align_strictly_before(g, indian_dates, name)
    assert (lag.dropna() >= 1).all(), \
        f"{name}: a global observation dated ON the Indian date was used"
    return True


# ---------------------------------------------------------------------
# DATE PARSING
# ---------------------------------------------------------------------
_EXPLICIT_FORMATS = (
    "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d",
    "%d-%m-%Y %H:%M:%S", "%d-%m-%Y %H:%M", "%d-%m-%Y",
    "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y",
    "%d-%b-%Y %H:%M:%S", "%d-%b-%Y %H:%M", "%d-%b-%Y",
)


def parse_dates(s, name="", verbose=False):
    """Parse a date column of unknown convention, inferring which applies.

    Returns (parsed, report). The report records the convention chosen and how
    many rows the rejected convention would have kept, so a silent switch is
    impossible to miss in the log.
    """
    s = pd.Series(s)
    n = len(s)
    a = pd.to_datetime(s, dayfirst=True, errors="coerce")
    b = pd.to_datetime(s, dayfirst=False, errors="coerce")
    na, nb = int(a.notna().sum()), int(b.notna().sum())

    if na >= nb:
        out, conv, loser = a.copy(), "day-first", nb
    else:
        out, conv, loser = b.copy(), "ISO/month-first", na

    for fmt in _EXPLICIT_FORMATS:
        miss = out.isna()
        if not miss.any():
            break
        out.loc[miss] = pd.to_datetime(s[miss], format=fmt, errors="coerce")

    bad = int(out.isna().sum())
    rep = {"name": name, "rows": n, "convention": conv,
           "parsed": n - bad, "unparsed": bad, "other_would_keep": loser}
    if verbose:
        print(f"  {name:26} {conv:16} {n-bad:>9,}/{n:<9,}"
              f"   (other convention: {loser:,})")
    return out, rep


def _assert_parser_correct():
    """Prove the parser handles both conventions WITHOUT transposing.

    The second case is the one that matters: an ISO column must not be read
    day-first. A parser that merely avoids NaT can still silently swap day and
    month, which is the failure this whole cell exists to prevent.
    """
    iso = pd.Series(["2015-01-09 09:15:00", "2015-01-13 09:15:00",
                     "2015-09-01 09:15:00", "2015-11-25 09:15:00"])
    got, _ = parse_dates(iso)
    assert list(got.dt.month) == [1, 1, 9, 11], f"ISO month wrong: {list(got.dt.month)}"
    assert list(got.dt.day) == [9, 13, 1, 25], f"ISO day wrong: {list(got.dt.day)}"

    dmy = pd.Series(["12-01-2015 09:15", "13-01-2015 09:20",
                     "01-09-2015 09:25", "25-11-2015 09:30"])
    got2, _ = parse_dates(dmy)
    assert list(got2.dt.month) == [1, 1, 9, 11], f"D-M-Y month wrong: {list(got2.dt.month)}"
    assert list(got2.dt.day) == [12, 13, 1, 25], f"D-M-Y day wrong: {list(got2.dt.day)}"

    print("  [PASS] both conventions parsed with no day/month transposition")
    return True


# ---------------------------------------------------------------------
# LOADERS
# ---------------------------------------------------------------------
def load_intraday_csv(path, session_start=SESSION_START, session_end=SESSION_END,
                      min_bars=MIN_BARS_PER_DAY, dayfirst=None,
                      max_unparsed_frac=0.01, min_expected_days=None):
    """Five-minute OHLCV loader.

    `dayfirst` is accepted for signature compatibility and IGNORED; the
    convention is inferred. Two guards are applied: an unparsed-share ceiling,
    and an optional floor on distinct trading days. The second matters because
    the transposition failure produces ZERO unparsed rows while roughly halving
    the day count, so a completeness check alone would pass it.
    """
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    dt_col = next(c for c in df.columns if c in ("date", "datetime", "timestamp"))

    n_raw = len(df)
    df["datetime"], rep = parse_dates(df[dt_col], os.path.basename(path))
    n_bad = rep["unparsed"]

    if n_bad > max_unparsed_frac * n_raw:
        raise ValueError(
            f"{os.path.basename(path)}: {n_bad:,} of {n_raw:,} timestamps "
            f"({100*n_bad/n_raw:.1f}%) unparsed under the {rep['convention']} "
            f"convention. Inspect the file; a run on the remainder would be a "
            f"run on a calendar-dependent subsample. Examples: "
            f"{df.loc[df['datetime'].isna(), dt_col].head(3).tolist()}")

    df = df.dropna(subset=["datetime"]).sort_values("datetime")

    t = df["datetime"].dt.time
    in_sess = ((t >= pd.Timestamp(session_start).time()) &
               (t <= pd.Timestamp(session_end).time()))
    n_out = int((~in_sess).sum())
    df = df[in_sess].copy()

    df["date"] = df["datetime"].dt.normalize()
    cnt = df.groupby("date")["close"].transform("size")
    n_stub = int((cnt < min_bars).sum())
    df = df[cnt >= min_bars].reset_index(drop=True)

    n_days = df["date"].nunique()
    if min_expected_days is not None and n_days < min_expected_days:
        raise ValueError(
            f"{os.path.basename(path)}: only {n_days:,} distinct trading days "
            f"(expected at least {min_expected_days:,}). A day count far below "
            f"expectation with no unparsed rows is the signature of transposed "
            f"day/month values. Convention chosen: {rep['convention']}.")

    print(f"  {os.path.basename(path)}: {len(df):,} bars kept | {n_bad} unparsed, "
          f"{n_out} out-of-session, {n_stub} stub-day bars dropped | "
          f"{n_days:,} days [{rep['convention']}]")
    return df


def load_us_daily(path):
    """US daily file: date, SP500, VIX (closes)."""
    d = pd.read_csv(path)
    d.columns = [c.strip().lower() for c in d.columns]
    dcol = next(c for c in d.columns if c in ("date", "datetime"))
    d[dcol], rep = parse_dates(d[dcol], os.path.basename(path))
    d = d.dropna(subset=[dcol]).sort_values(dcol).set_index(dcol)
    out = {}
    if "sp500" in d.columns:
        out["SPX"] = d[["sp500"]].rename(columns={"sp500": "close"})
    if "vix" in d.columns:
        out["VIX_US"] = d[["vix"]].rename(columns={"vix": "close"})
    for k, v in out.items():
        print(f"  [{k}] {len(v):,} rows  {v.index.min().date()} -> "
              f"{v.index.max().date()} [{rep['convention']}]")
    return out


def load_global_csv(path, name):
    """WTI or USD/INR. Tolerant of long-form FX (one row per country per date)."""
    d = pd.read_csv(path)
    d.columns = [c.strip().lower() for c in d.columns]
    dcol = next(c for c in d.columns if c in ("date", "datetime"))
    d[dcol], rep = parse_dates(d[dcol], f"{name} ({os.path.basename(path)})")
    d = d.dropna(subset=[dcol]).sort_values(dcol)

    if name == "FX" and "country" in d.columns:
        d = d[d["country"].astype(str).str.contains("India", case=False, na=False)]
        if d.empty:
            raise ValueError(f"{path}: no rows matching country 'India'")

    val = next(c for c in d.columns if c not in (dcol, "country"))
    col = "usdinr" if name == "FX" else "price"
    d = d[[dcol, val]].rename(columns={val: col}).set_index(dcol)
    d = d[~d.index.duplicated(keep="last")]

    print(f"  [{name}] {len(d):,} rows  {d.index.min().date()} -> "
          f"{d.index.max().date()}   (column '{val}' -> '{col}') "
          f"[{rep['convention']}]")
    return d


def load_all(data_dir=DATA_DIR_LOCAL, index_file="NIFTY PSU BANK_5minute.csv",
             min_expected_days=1500):
    """Read every input from disk. No network access anywhere in this notebook.

    `min_expected_days` guards the index series against the transposition
    failure. Set it below the shortest history you expect: the sectoral and
    large-cap files carry roughly 2,800 trading days and the shorter size
    indices roughly 1,700, so 1,500 is a safe floor that still catches a
    collapse to ~1,100.
    """
    print(f"Reading all inputs from ./{data_dir}/   (no network access)\n")
    _assert_parser_correct()

    nifty_raw = load_intraday_csv(os.path.join(data_dir, index_file),
                                  min_expected_days=min_expected_days)
    vix_raw = load_intraday_csv(os.path.join(data_dir, "INDIA VIX_5minute.csv"),
                                min_expected_days=min_expected_days)

    G = {}
    us_path = os.path.join(data_dir, "us_daily.csv")
    if os.path.exists(us_path):
        G.update(load_us_daily(us_path))
    else:
        print(f"  [SPX/VIX_US] MISSING: {us_path}")

    for nm, fn in (("WTI", "WTI.csv"), ("FX", "FX.csv")):
        p = os.path.join(data_dir, fn)
        if os.path.exists(p):
            G[nm] = load_global_csv(p, nm)
        else:
            print(f"  [{nm}] MISSING: {p} - the {nm} feature block will be omitted")

    missing = [k for k in ("SPX", "VIX_US", "WTI", "FX") if k not in G]
    if missing:
        print(f"\n  STILL MISSING: {missing} - affected blocks are dropped and every "
              f"downstream table reports the omission")
    return nifty_raw, vix_raw, G, missing


In [3]:
# Feature Engineering
JUMP_Z = stats.norm.ppf(JUMP_ALPHA)

def realized_measures(r):
    M = len(r)
    if M < MIN_BARS_PER_DAY:
        return None
    RV = float((r ** 2).sum())
    BV = float((np.pi / 2) * np.sum(np.abs(r[:-1]) * np.abs(r[1:])) * M / max(M - 1, 1))
    RQ = float(M / 3 * np.sum(r ** 4))
    TQ = float(M * (2 ** (2 / 3) * gamma_fn(7 / 6) / gamma_fn(0.5)) ** -3 * np.sum((np.abs(r[:-2]) * np.abs(r[1:-1]) * np.abs(r[2:])) ** (4 / 3)))
    rel = (RV - BV) / RV if RV > 0 else 0.0
    den = np.sqrt(max(0.6090 * max(TQ / BV ** 2, 1.0) / M, 1e-16)) if BV > 0 else np.nan
    Z = rel / den if den and np.isfinite(den) and den > 0 else 0.0
    J = max(RV - BV, 0.0) if Z > JUMP_Z else 0.0
    return dict(RV=RV, BV=BV, RQ=RQ,  RS_plus=float((r[r > 0] ** 2).sum()), RS_minus=float((r[r < 0] ** 2).sum()), J=J, C=RV - J, RSkew=float(np.sqrt(M) * np.sum(r ** 3) / max(RV ** 1.5, 1e-30)), RKurt=float(M * np.sum(r ** 4) / max(RV ** 2, 1e-30)))

def build_daily_measures(intraday, first_bar_fix=True):
    d = intraday.copy()
    d["log_close"] = np.log(d["close"])
    d["ret"] = d.groupby("date")["log_close"].diff()
    if first_bar_fix and "open" in d.columns:
        first = d.groupby("date").cumcount() == 0
        d.loc[first, "ret"] = d.loc[first, "log_close"] - np.log(d.loc[first, "open"])
    d = d.dropna(subset=["ret"])
    recs = []
    for dt, g in d.groupby("date"):
        m = realized_measures(g["ret"].values)
        if m:
            m["date"] = dt
            m["close"] = float(g["close"].iloc[-1])
            m["open"] = float(g["open"].iloc[0]) if "open" in g.columns else np.nan
            recs.append(m)
    return pd.DataFrame(recs).set_index("date").sort_index()

def build_features(rv_table, G, missing, horizon=1):
    f = rv_table.copy()
    ind = f.index
    notes = []
    for c in ["RV", "C", "IV"]:
        lg = np.log(f[c].clip(lower=1e-16))
        for w in (1, 5, 22):
            f[f"log{c}_{w}"] = lg.rolling(w).mean()
    for w in (1, 5, 22):
        f[f"RV_{w}"] = f["RV"].rolling(w).mean()
    f["ret_d"] = np.log(f["close"]).diff()
    _rf = f["ret_d"].fillna(0.0)
    for w in (1, 5, 22):
        if w == 1:
            f["r_pos_1"], f["r_neg_1"] = _rf.clip(lower=0), _rf.clip(upper=0)
        else:
            f[f"r_pos_{w}"] = _rf.clip(lower=0).rolling(w).mean()
            f[f"r_neg_{w}"] = _rf.clip(upper=0).rolling(w).mean()
    _lrv = np.log(f["RV"].clip(lower=1e-16))
    for w in (5, 22):
        f[f"logVoV_{w}"] = np.log(_lrv.rolling(w).std().clip(lower=1e-8))
    f["SJV_ratio_1"] = (f["RS_minus"] - f["RS_plus"]) / f["RV"].clip(lower=1e-16)
    f["JumpShare_1"] = (f["J"] / f["RV"].clip(lower=1e-16)).clip(0, 1)
    f["logRQ_1"] = np.log(f["RQ"].clip(lower=1e-30))
    f["HARQ_inter_1"] = ((f["logRQ_1"] - f["logRQ_1"].expanding(min_periods=100).mean().shift(1)) * f["logRV_1"])
    f["logRV_curv"] = f["logRV_1"] - 2 * f["logRV_5"] + f["logRV_22"]
    f["VRP_1"] = f["logIV_1"] - f["logRV_1"]
    f["logIV_slope"] = f["logIV_1"] - f["logIV_22"]
    DOMESTIC = (["logRV_1", "logRV_5", "logRV_22", "logIV_1"] +  [f"r_{s}_{w}" for w in (1, 5, 22) for s in ("pos", "neg")])
    DOM_EXT = ["logVoV_5", "logVoV_22", "SJV_ratio_1", "JumpShare_1","logRQ_1", "HARQ_inter_1", "logIV_slope"]

    GLOBAL_IV = []
    if "VIX_US" in G:
        vx = G["VIX_US"]
        _have = [c for c in ("close", "open", "high", "low") if c in vx.columns]
        for col in _have:
            _s, _ = align_strictly_before(vx[col], ind, col)
            f["vix" + col[0]] = _s.values
        f["logVIXus_1"] = np.log(f["vixc"].clip(lower=1e-6))
        f["dlogVIXus"] = f["logVIXus_1"].diff()
        f["dlogVIXus_5"] = f["logVIXus_1"].diff(5)
        _has_ohlc = {"vixh", "vixl", "vixo"} <= set(f.columns)
        if _has_ohlc:
            f["VIXus_range"] = np.log(f["vixh"].clip(lower=1e-6) / f["vixl"].clip(lower=1e-6))
            f["VIXus_gap"] = np.log(f["vixo"].clip(lower=1e-6)) - f["logVIXus_1"].shift(1)
        _us_var = (f["vixc"] / 100.0) ** 2 / TRADING_DAYS_PER_YEAR
        f["us_logvar"] = np.log(_us_var.clip(lower=1e-16))    
        GLOBAL_IV = ["logVIXus_1", "dlogVIXus", "dlogVIXus_5"]
        if _has_ohlc:
            GLOBAL_IV += ["VIXus_range", "VIXus_gap"]
        else:
            notes.append("VIX source is close-only - VIXus_range/VIXus_gap omitted")
    else:
        notes.append("US VIX unavailable - GLOBAL_IV block omitted")
    GLOBAL_FX = []
    if "FX" in G:
        s, _ = align_strictly_before(G["FX"]["usdinr"], ind, "usdinr")
        f["usdinr"] = s.values
        f["r_inr"] = np.log(f["usdinr"]).diff()
        f["r_inr_pos"] = f["r_inr"].clip(lower=0)        
        f["r_inr_neg"] = f["r_inr"].clip(upper=0)
        f["inr_vol_22"] = f["r_inr"].rolling(22).std()
        GLOBAL_FX = ["r_inr_pos", "r_inr_neg", "inr_vol_22"]
    else:
        notes.append("USD/INR unavailable - GLOBAL_FX block omitted")

    GLOBAL_CMD = []
    if "WTI" in G:
        s, _ = align_strictly_before(G["WTI"]["price"], ind, "wti")
        _w = s.copy()
        _n_neg = int((_w <= 0).sum())
        _w[_w <= 0] = np.nan
        f["wti"] = _w.values
        f["r_wti"] = np.log(f["wti"]).diff()
        if _n_neg:
            print(f"WTI: {_n_neg} non-positive settlement(s) set to NaN "
                  f"(negative-price artifact)")
        f["r_wti_pos"] = f["r_wti"].clip(lower=0)
        f["r_wti_neg"] = f["r_wti"].clip(upper=0)
        f["wti_vol_22"] = f["r_wti"].rolling(22).std()
        GLOBAL_CMD = ["r_wti_pos", "r_wti_neg", "wti_vol_22"]
    else:
        notes.append("WTI unavailable - GLOBAL_CMD block omitted")

    GLOBAL_EQ = []
    if "SPX" in G:
        sp = G["SPX"]
        cc = "close" if "close" in sp.columns else sp.columns[-1]
        s, _ = align_strictly_before(sp[cc], ind, "spx")
        f["spx"] = s.values
        f["r_spx"] = np.log(f["spx"].clip(lower=1e-6)).diff()
        f["r_spx_pos"], f["r_spx_neg"] = f["r_spx"].clip(lower=0), f["r_spx"].clip(upper=0)
        f["spx_vol_22"] = f["r_spx"].rolling(22).std()
        GLOBAL_EQ = ["r_spx_pos", "r_spx_neg", "spx_vol_22"]
        if {"high", "low"} <= set(sp.columns):
            hs, _ = align_strictly_before(sp["high"], ind, "spxh")
            ls, _ = align_strictly_before(sp["low"], ind, "spxl")
            f["spx_park"] = (np.log(hs.values / ls.values) ** 2) / (4 * np.log(2))
            f["log_spx_park"] = np.log(f["spx_park"].clip(lower=1e-12))
            GLOBAL_EQ.append("log_spx_park")
    else:
        notes.append("S&P 500 unavailable - GLOBAL_EQ block omitted " "(dlogVIXus is its partial proxy: VIX and S&P returns " "correlate about -0.7)")
    INTERACT = []
    if GLOBAL_IV:
        f["ix_dVIX_state"] = f["dlogVIXus"] * f["logRV_1"]
        INTERACT.append("ix_dVIX_state")
    if GLOBAL_FX:
        f["ix_inr_state"] = f["r_inr_pos"] * f["logRV_1"]
        INTERACT.append("ix_inr_state")
    if GLOBAL_IV and GLOBAL_FX:
        f["ix_vix_inr"] = f["dlogVIXus"] * f["r_inr_pos"]
        INTERACT.append("ix_vix_inr")
    f["RV_target"] = f["RV"].shift(-1).rolling(horizon).mean().shift(-(horizon - 1))
    f["logRV_target"] = np.log(f["RV_target"].clip(lower=1e-16))
    GLOBAL_ALL = GLOBAL_IV + GLOBAL_FX + GLOBAL_CMD + GLOBAL_EQ
    BLOCKS = {"DOMESTIC": DOMESTIC, "DOM_EXT": DOM_EXT, "GLOBAL_IV": GLOBAL_IV, "GLOBAL_FX": GLOBAL_FX, "GLOBAL_CMD": GLOBAL_CMD, "GLOBAL_EQ": GLOBAL_EQ, "GLOBAL_ALL": GLOBAL_ALL, "INTERACT": INTERACT}
    need = sorted({c for v in BLOCKS.values() for c in v})
    feat = f.dropna(subset=need).copy()
    assert_blocks_full_rank(feat, BLOCKS)
    return feat, BLOCKS, notes


def assert_blocks_full_rank(feat, BLOCKS, tol=1e-8):
    bad = []
    for name, cols in BLOCKS.items():
        if len(cols) < 2:
            continue
        Z = feat[cols].dropna().values.astype(float)
        r = np.linalg.matrix_rank(Z, tol=tol * max(1.0, np.abs(Z).max()))
        if r < len(cols):
            bad.append((name, len(cols), r))
    allc = sorted({c for v in BLOCKS.values() for c in v})
    Z = feat[allc].dropna().values.astype(float)
    r = np.linalg.matrix_rank(Z)
    if r < len(allc):
        bad.append(("UNION", len(allc), r))
    if bad:
        msg = "; ".join(f"{n}: rank {r} of {k}" for n, k, r in bad)
        raise AssertionError(
            f"exact linear dependence in the feature blocks -> {msg}. "
            "Some column is a linear combination of the others; drop it or "
            "re-parameterise the block before any test is run.")
    return True


def feature_sets(BLOCKS):
    D, DX = BLOCKS["DOMESTIC"], BLOCKS["DOM_EXT"]
    S = {
        "DOM":            D,                                  # the Part I benchmark
        "DOM_PLUS":       D + DX,                             # richer domestic
        "DOM_GIV":        D + BLOCKS["GLOBAL_IV"],            # + US implied vol
        "DOM_GFX":        D + BLOCKS["GLOBAL_FX"],            # + FX
        "DOM_GCMD":       D + BLOCKS["GLOBAL_CMD"],           # + commodity
        "DOM_GLOBAL":     D + BLOCKS["GLOBAL_ALL"],           # + all global
        "DOM_GLOBAL_IX":  D + BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"],
        "FULL":           D + DX + BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"],
    }
    return {k: [c for c in v if c] for k, v in S.items() if v}

In [4]:
# Purged Walk forward
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
try:
    from xgboost import XGBRegressor; HAS_XGB = True
except Exception: HAS_XGB = False
try:
    from lightgbm import LGBMRegressor; HAS_LGBM = True
except Exception: HAS_LGBM = False

CV_FOLDS       = 4
EMBARGO_FRAC   = 0.01     
ML_REFIT_EVERY = 21
TUNE_EVERY     = 252

def make_model(name, seed=0, **hp):
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=hp.get("alpha", 1.0)))
    if name == "Lasso":
        return make_pipeline(StandardScaler(), Lasso(alpha=hp.get("alpha", 1e-3), max_iter=5000))
    if name == "ElasticNet":
        return make_pipeline(StandardScaler(), ElasticNet(alpha=hp.get("alpha", 1e-3), l1_ratio=hp.get("l1_ratio", 0.5), max_iter=5000))
    if name == "RandomForest":
        return RandomForestRegressor(n_estimators=200, max_depth=hp.get("max_depth"), min_samples_leaf=hp.get("min_samples_leaf", 20), max_features=hp.get("max_features", 0.5), random_state=seed, n_jobs=1)
    if name == "ExtraTrees":
        return ExtraTreesRegressor(n_estimators=200, max_depth=hp.get("max_depth"),  min_samples_leaf=hp.get("min_samples_leaf", 20), max_features=hp.get("max_features", 0.5), random_state=seed, n_jobs=1)
    if name == "GBM":
        return HistGradientBoostingRegressor(learning_rate=hp.get("learning_rate", 0.05), max_depth=hp.get("max_depth", 3), min_samples_leaf=hp.get("min_samples_leaf", 20), l2_regularization=hp.get("l2", 1.0), max_iter=hp.get("max_iter", 300), early_stopping=False, random_state=seed)
    if name == "XGB":
        return XGBRegressor(n_estimators=hp.get("n_estimators", 400),   learning_rate=hp.get("learning_rate", 0.03),  max_depth=hp.get("max_depth", 3),  min_child_weight=hp.get("min_child_weight", 20), subsample=0.8, colsample_bytree=0.8, reg_lambda=hp.get("reg_lambda", 1.0),  random_state=seed, n_jobs=1, tree_method="hist", verbosity=0)
    if name == "LGBM":
        return LGBMRegressor(n_estimators=hp.get("n_estimators", 400), learning_rate=hp.get("learning_rate", 0.03), num_leaves=hp.get("num_leaves", 15), min_child_samples=hp.get("min_child_samples", 20), subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=hp.get("reg_lambda", 1.0), random_state=seed, n_jobs=1, verbose=-1)
    if name == "MLP":
        return make_pipeline(StandardScaler(),
                             MLPRegressor(hidden_layer_sizes=hp.get("hidden", (16,)), alpha=hp.get("alpha", 1e-2), max_iter=1200,  early_stopping=False, random_state=seed))
    if name == "SVR":
        return make_pipeline(StandardScaler(), SVR(kernel="rbf", C=hp.get("C", 1.0),  epsilon=hp.get("epsilon", 0.1)))
    if name == "OLS":
        return make_pipeline(StandardScaler(), LinearRegression())
    raise ValueError(name)


GRIDS = {
 "Ridge":[{"alpha": a} for a in [0.1, 1.0, 10.0, 100.0]],
 "Lasso": [{"alpha": a} for a in [1e-4, 1e-3, 1e-2]],
 "ElasticNet": [{"alpha": a, "l1_ratio": l} for a in [1e-3, 1e-2] for l in [0.5, 0.9]],
 "RandomForest":[{"max_depth": d, "max_features": m} for d in [4, 8] for m in [0.33, 0.66]],
 "ExtraTrees":[{"max_depth": d, "max_features": m} for d in [4, 8] for m in [0.33, 0.66]],
 "GBM":[{"learning_rate": lr, "max_depth": d, "l2": l2} for lr in [0.03, 0.05] for d in [2, 3] for l2 in [1.0, 10.0]],
 "XGB":[{"max_depth": d, "reg_lambda": rl, "min_child_weight": m} for d in [2, 3] for rl in [1.0, 10.0] for m in [10, 30]],               
 "LGBM":[{"num_leaves": nl, "reg_lambda": rl} for nl in [7, 15] for rl in [1.0, 10.0]],
 "MLP":[{"hidden": h, "alpha": a} for h in [(8,), (16,), (16, 8)] for a in [1e-2, 1e-1]],
 "SVR":[{"C": c, "epsilon": e} for c in [1.0, 10.0] for e in [0.05, 0.2]],
}
STOCHASTIC = {"RandomForest", "ExtraTrees", "GBM", "XGB", "LGBM", "MLP"}
ALL_MODELS = [m for m in ["Ridge", "Lasso", "ElasticNet", "RandomForest", "ExtraTrees","GBM", "XGB", "LGBM", "MLP", "SVR"] if (m != "XGB" or HAS_XGB) and (m != "LGBM" or HAS_LGBM)]

def purged_folds(n, h, n_folds=CV_FOLDS, embargo_frac=EMBARGO_FRAC):
    emb = max(1, int(embargo_frac * n))
    block = n // (n_folds + 1)
    folds = []
    for k in range(1, n_folds + 1):
        v0, v1 = block * k, (block * (k + 1) if k < n_folds else n)
        if v1 - v0 < 20:
            continue
        train = np.arange(0, max(0, v0 - h))            
        val = np.arange(v0, v1)
        folds.append((train, val, (v1, min(n, v1 + emb))))  
    return folds

def purged_cv_score(model_name, hp, X, y, seed, h, n_folds=CV_FOLDS):
    n = len(X)
    if n < 250:
        return np.inf
    errs = []
    for tr, va, _ in purged_folds(n, h, n_folds):
        if len(tr) < 150 or len(va) < 20:
            continue
        m = make_model(model_name, seed=seed, **hp)
        m.fit(X[tr], y[tr])
        errs.append(float(np.mean((m.predict(X[va]) - y[va]) ** 2)))
    return float(np.mean(errs)) if errs else np.inf

def assert_folds_purged(n=1000, h=5):
    for tr, va, emb in purged_folds(n, h):
        if len(tr) == 0:
            continue
        assert tr.max() < va.min() - h + 1, \
            f"purge failed: train max {tr.max()} too close to val start {va.min()} (h={h})"
    return True

def qlike(a, p):
    r = np.asarray(a) / np.asarray(p)
    return r - np.log(r) - 1

def run_walkforward(feat, cols, model_names, bench_cols, h=1, seeds=SEEDS, oos_fraction=OOS_FRACTION, refit_every=ML_REFIT_EVERY,  tune_every=TUNE_EVERY, verbose=True):
    usable = feat[feat["RV_target"].notna()]
    n0 = int(len(usable) * (1 - oos_fraction))
    oos = usable.index[n0:]
    X_all = feat[cols].values.astype(float)
    y_all = feat["logRV_target"].values.astype(float)
    idx_of = {d: i for i, d in enumerate(feat.index)}
    preds = {f"{m}|s{s}": [] for m in model_names for s in seeds}
    preds["HAR_bench"] = []
    actuals, origins, state, best_hp, hp_log = [], [], {}, {}, []
    t0 = time.time()
    for i, t in enumerate(oos):
        pos = idx_of[t]; end = pos - h + 1
        if end < 300:
            continue
        tr = feat.iloc[:end]; tr = tr[tr["RV_target"].notna()]
        if len(tr) < 300:
            continue
        y_true = feat["RV_target"].iloc[pos]
        if not np.isfinite(y_true) or y_true <= 0:
            continue
        actuals.append(y_true); origins.append(t)
        Xtr = tr[cols].values.astype(float); ytr = tr["logRV_target"].values.astype(float)
        xnew = X_all[pos].reshape(1, -1)
        floor = max(PRED_FLOOR_FRAC * float(tr["RV"].median()), 1e-16)
        Xb = sm.add_constant(tr[bench_cols].values)
        rb = sm.OLS(ytr, Xb).fit()
        mub = float(rb.predict(np.r_[1.0, feat[bench_cols].iloc[pos].values].reshape(1, -1))[0])
        preds["HAR_bench"].append(max(np.exp(mub + 0.5 * np.var(rb.resid)), floor))
        for mn in model_names:
            use_seeds = seeds if mn in STOCHASTIC else (seeds[0],)
            for s in use_seeds:
                key = (mn, s)
                if (i % tune_every == 0) or (key not in best_hp):
                    sc = [(purged_cv_score(mn, hp, Xtr, ytr, s, h), hp) for hp in GRIDS[mn]]
                    best_hp[key] = min(sc, key=lambda z: z[0])[1]
                    hp_log.append({"origin": t, "model": mn, "seed": s, **best_hp[key]})
                if (i % refit_every == 0) or (key not in state):
                    m = make_model(mn, seed=s, **best_hp[key]); m.fit(Xtr, ytr)
                    state[key] = (m, float(np.var(ytr - m.predict(Xtr))))
                m, sig2 = state[key]
                preds[f"{mn}|s{s}"].append(
                    max(np.exp(float(m.predict(xnew)[0]) + 0.5 * sig2), floor))
            if mn not in STOCHASTIC:
                base = preds[f"{mn}|s{seeds[0]}"][-1]
                for s in seeds[1:]:
                    preds[f"{mn}|s{s}"].append(base)
        if verbose and i % 200 == 0 and i > 0:
            print(f"    origin {i}/{len(oos)}  ({time.time()-t0:.0f}s)")

    out = pd.DataFrame({k: v for k, v in preds.items() if len(v) == len(actuals)}, index=pd.DatetimeIndex(origins))
    out["actual"] = actuals         
    return out, pd.DataFrame(hp_log)

In [5]:
# ML on HAR Residuals Using Global features
def run_har_boost(feat, global_cols, bench_cols, model_names, h=1, seeds=(0, 1, 2), oos_fraction=OOS_FRACTION, refit_every=ML_REFIT_EVERY, tune_every=TUNE_EVERY, shrink=1.0, verbose=False):
    usable = feat[feat["RV_target"].notna()]
    n0 = int(len(usable) * (1 - oos_fraction))
    oos = usable.index[n0:]
    Xg = feat[global_cols].values.astype(float)
    idx_of = {d: i for i, d in enumerate(feat.index)}

    preds = {f"BOOST_{m}|s{s}": [] for m in model_names for s in seeds}
    preds["HAR_bench"] = []
    actuals, origins, state, best_hp = [], [], {}, {}
    t0 = time.time()
    for i, t in enumerate(oos):
        pos = idx_of[t]; end = pos - h + 1
        if end < 300:
            continue
        tr = feat.iloc[:end]; tr = tr[tr["RV_target"].notna()]
        if len(tr) < 300:
            continue
        y_true = feat["RV_target"].iloc[pos]
        if not np.isfinite(y_true) or y_true <= 0:
            continue
        actuals.append(y_true); origins.append(t)
        ytr = tr["logRV_target"].values.astype(float)
        floor = max(PRED_FLOOR_FRAC * float(tr["RV"].median()), 1e-16)

        Xb = sm.add_constant(tr[bench_cols].values)
        rb = sm.OLS(ytr, Xb).fit()
        sig2 = float(np.var(rb.resid))
        mub = float(rb.predict(np.r_[1.0, feat[bench_cols].iloc[pos].values].reshape(1, -1))[0])
        preds["HAR_bench"].append(max(np.exp(mub + 0.5 * sig2), floor))

        e_tr = ytr - rb.fittedvalues
        Xg_tr = feat.loc[tr.index, global_cols].values.astype(float)
        xg_new = Xg[pos].reshape(1, -1)
        for mn in model_names:
            use_seeds = seeds if mn in STOCHASTIC else (seeds[0],)
            for s in use_seeds:
                key = (mn, s)
                if (i % tune_every == 0) or (key not in best_hp):
                    sc = [(purged_cv_score(mn, hp, Xg_tr, e_tr, s, h), hp) for hp in GRIDS[mn]]
                    best_hp[key] = min(sc, key=lambda z: z[0])[1]
                if (i % refit_every == 0) or (key not in state):
                    m = make_model(mn, seed=s, **best_hp[key]); m.fit(Xg_tr, e_tr)
                    state[key] = m
                m = state[key]
                adj = shrink * float(m.predict(xg_new)[0])
                preds[f"BOOST_{mn}|s{s}"].append(max(np.exp(mub + adj + 0.5 * sig2), floor))
            if mn not in STOCHASTIC:
                base = preds[f"BOOST_{mn}|s{seeds[0]}"][-1]
                for s in seeds[1:]:
                    preds[f"BOOST_{mn}|s{s}"].append(base)
        if verbose and i % 200 == 0 and i > 0:
            print(f"boost origin {i}/{len(oos)} ({time.time()-t0:.0f}s)")

    out = pd.DataFrame({k: v for k, v in preds.items() if len(v) == len(actuals)},index=pd.DatetimeIndex(origins))
    out["actual"] = actuals
    return out

In [6]:
# Diebold Mariano
def dm(l1, l2, h=1):
    d = np.asarray(l1) - np.asarray(l2)
    scale = max(float(np.mean(np.abs(l1))), 1e-300)
    if np.max(np.abs(d)) < 1e-10 * scale:
        return 0.0, 1.0
    d = np.asarray(l1) - np.asarray(l2); n = len(d)
    g0 = np.var(d, ddof=0)
    cov = sum(2 * (1 - k / h) * np.cov(d[:-k], d[k:])[0, 1] for k in range(1, h) if k < n)
    v = (g0 + cov) / n
    if not np.isfinite(v) or v <= 0:
        return np.nan, np.nan
    t = d.mean() / np.sqrt(v) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return t, 2 * (1 - stats.t.cdf(abs(t), df=n - 1))

# Benjamini Hochberg 
def benjamini_hochberg(pvals, names, alpha=0.10):
    p = np.asarray(pvals, float); ok = np.isfinite(p)
    idx = np.argsort(np.where(ok, p, np.inf)); m = int(ok.sum())
    if m == 0:
        return set()
    th = (np.arange(1, m + 1) / m) * alpha
    passed = p[idx][:m] <= th
    kmax = int(np.max(np.where(passed)[0])) + 1 if passed.any() else 0
    return {names[i] for i in idx[:kmax]}

# Clark West Test
def clark_west(y, f_r, f_u, h=1):
    y = np.asarray(y); fr = np.asarray(f_r); fu = np.asarray(f_u)
    adj = (y - fr) ** 2 - ((y - fu) ** 2 - (fr - fu) ** 2)
    r = sm.OLS(adj, np.ones(len(adj))).fit(cov_type="HAC", cov_kwds={"maxlags": int(max(1, h))})                                         
    return float(r.params[0]), float(r.tvalues[0]), float(r.pvalues[0])

# MCS Test
def model_confidence_set(L, alpha=0.10, n_boot=1000, block=10, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    names = list(L.columns); n = len(L)
    nb = int(np.ceil(n / block))
    st = rng.integers(0, n, size=(n_boot, nb))
    bi = np.concatenate([st[:, j:j + 1] + np.arange(block) for j in range(nb)], axis=1)[:, :n] % n
    surv, elim = list(names), []
    while len(surv) > 1:
        M = L[surv].values; k = len(surv)
        db = (M[:, :, None] - M[:, None, :]).mean(0).sum(1) / (k - 1)
        bt = np.empty((n_boot, k))
        for b in range(n_boot):
            Lb = M[bi[b]]
            bt[b] = (Lb[:, :, None] - Lb[:, None, :]).mean(0).sum(1) / (k - 1)
        v = bt.var(0, ddof=1); v[v <= 0] = np.nan
        ti = db / np.sqrt(v); tb = (bt - db) / np.sqrt(v)
        p = float(np.mean(np.nanmax(np.abs(tb), 1) >= np.nanmax(np.abs(ti))))
        if p > alpha or not np.isfinite(ti).any():
            break
        w = surv[int(np.nanargmax(ti))]; elim.append((w, p)); surv.remove(w)
    return surv, elim

# Markowitz test
def mz_recalibrate(actual, pred, warmup=250):
    a = np.log(np.clip(np.asarray(actual, float), 1e-300, None))
    p = np.log(np.clip(np.asarray(pred, float), 1e-300, None))
    n = len(a); out = np.full(n, np.nan); active = np.zeros(n, bool)
    for i in range(n):
        if i < warmup:
            out[i] = p[i]; continue
        A, P = a[:i], p[:i]                     
        ok = np.isfinite(A) & np.isfinite(P)
        if ok.sum() < warmup:
            out[i] = p[i]; continue
        X = np.column_stack([np.ones(ok.sum()), P[ok]])
        try:
            b = np.linalg.lstsq(X, A[ok], rcond=None)[0]
        except Exception:
            out[i] = p[i]; continue
        out[i] = b[0] + b[1] * p[i]; active[i] = True
    return np.exp(out), active

def mz_efficiency(actual, pred, h=1):
    a = np.log(np.clip(np.asarray(actual, float), 1e-300, None))
    p = np.log(np.clip(np.asarray(pred, float), 1e-300, None))
    ok = np.isfinite(a) & np.isfinite(p)
    r = sm.OLS(a[ok], sm.add_constant(p[ok])).fit(cov_type="HAC",cov_kwds={"maxlags": int(max(1, h))})                                              
    f = r.f_test((np.eye(2), np.array([[0.0], [1.0]])))
    return {"a": float(r.params[0]), "b": float(r.params[1]),"R2": float(r.rsquared), "p": float(f.pvalue)}

# Giacomini Rossi Fluctuation test
def giacomini_rossi(l_bench, l_model, frac=0.30, h=1):
    d = np.asarray(l_bench) - np.asarray(l_model)
    T = len(d); m = max(30, int(frac * T))
    if T <= m + 5:
        return None
    S = []
    for s in range(T - m + 1):
        w = d[s:s + m]
        v = np.var(w, ddof=0) / m
        S.append(w.mean() / np.sqrt(v) if v > 0 else np.nan)
    S = np.asarray(S)
    CV = {0.1: 3.393, 0.2: 3.170, 0.3: 3.012, 0.5: 2.777}
    cv = CV.get(round(frac, 1), 3.012)
    return {"S": S, "sup": float(np.nanmax(np.abs(S))), "cv5": cv,"unstable": bool(np.nanmax(np.abs(S)) > cv), "frac_favouring_model": float(np.nanmean(S > 0))}

def volatility_timing(ret_next, var_fc, gamma=3.0, target_vol=0.15, cost_bps=5.0, cap=2.0):
    r = np.asarray(ret_next, float); v = np.asarray(var_fc, float)
    ok = np.isfinite(r) & np.isfinite(v) & (v > 0)
    r, v = r[ok], v[ok]
    w = np.clip((target_vol ** 2 / TRADING_DAYS_PER_YEAR) / (gamma * v), 0, cap)
    turn = np.abs(np.diff(np.r_[0.0, w]))
    rp = w * r - turn * cost_bps / 1e4
    ann = TRADING_DAYS_PER_YEAR
    mu, sd = rp.mean() * ann, rp.std() * np.sqrt(ann)
    return {"mean": mu, "vol": sd, "sharpe": mu / sd if sd > 0 else np.nan,"turnover": turn.mean() * ann, "w_mean": w.mean(),"util": mu - 0.5 * gamma * sd ** 2, "rp": rp, "w": w}

def performance_fee(res_a, res_b, gamma=3.0):
    return 1e4 * (res_a["util"] - res_b["util"])

In [7]:
# SHAP Inference
import shap

SHAP_BLOCK = 22         
SHAP_NULL_B = 200        
SHAP_BACKGROUND = 200   
SHAP_INTER_ROWS = 250    
SHAP_INTER_B = 30       

def _circular_block_permute(y, block, rng):
    n = len(y)
    nb = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=nb)
    out = np.concatenate([np.take(y, range(s, s + block), mode="wrap") for s in starts])
    return out[:n]

def _explainer_mean_abs(model, X_bg, X_eval, model_name):
    if model_name in ("RandomForest", "ExtraTrees", "GBM", "XGB", "LGBM"):
        ex = shap.TreeExplainer(model)
        sv = ex.shap_values(X_eval, check_additivity=False)
    elif model_name in ("Ridge", "Lasso", "ElasticNet"):
        ex = shap.LinearExplainer(model, X_bg)
        sv = ex.shap_values(X_eval)
    else:
        ex = shap.PermutationExplainer(model.predict, X_bg)
        sv = ex(X_eval, max_evals=2 * X_eval.shape[1] + 1, silent=True).values
    sv = np.asarray(sv)
    if sv.ndim == 3:                    
        sv = sv[..., 0]
    return np.abs(sv).mean(axis=0), sv

def shap_with_null(model_name, X_tr, y_tr, X_eval, feature_names, hp=None,seed=RNG_SEED, B=SHAP_NULL_B, block=SHAP_BLOCK, verbose=False):
    hp = hp or {}
    m = make_model(model_name, seed=seed, **hp); m.fit(X_tr, y_tr)
    bg = shap.utils.sample(X_tr, min(SHAP_BACKGROUND, len(X_tr)), random_state=seed)
    obs, sv = _explainer_mean_abs(m, bg, X_eval, model_name)
    rng = np.random.default_rng(seed)
    null = np.empty((B, X_tr.shape[1]))
    t0 = time.time()
    for b in range(B):
        yb = _circular_block_permute(y_tr, block, rng)
        mb = make_model(model_name, seed=seed + 1 + b, **hp); mb.fit(X_tr, yb)
        null[b], _ = _explainer_mean_abs(mb, bg, X_eval, model_name)
        if verbose and (b + 1) % 50 == 0:
            print(f"null {b+1}/{B}  ({time.time()-t0:.0f}s)", flush=True)

    p = (1 + (null >= obs).sum(axis=0)) / (B + 1)
    out = pd.DataFrame({"feature": feature_names,"mean|SHAP|": obs,"null mean": null.mean(axis=0),"null q95": np.quantile(null, 0.95, axis=0),"ratio": obs / np.maximum(null.mean(axis=0), 1e-30),"perm p": p,}).sort_values("mean|SHAP|", ascending=False).reset_index(drop=True)
    return out, sv, null

def _state_ratio(sv, turb, blocks, feature_names):
    A = np.abs(sv); out = {}
    for b, cols in blocks.items():
        idx = [feature_names.index(c) for c in cols if c in feature_names]
        if not idx:
            continue
        m = A[:, idx].sum(axis=1)
        out[b] = float(m[turb].mean() / max(m[~turb].mean(), 1e-30))
    return out

def shap_conditional_null(model_name, X_tr, y_tr, X_eval, feature_names, test_cols, hp=None, seed=RNG_SEED, B=SHAP_NULL_B, block=SHAP_BLOCK, verbose=False, state_mask=None, state_blocks=None):
    hp = hp or {}
    idx = [feature_names.index(c) for c in test_cols]
    m = make_model(model_name, seed=seed, **hp); m.fit(X_tr, y_tr)
    bg = shap.utils.sample(X_tr, min(SHAP_BACKGROUND, len(X_tr)), random_state=seed)
    obs, sv_obs = _explainer_mean_abs(m, bg, X_eval, model_name)
    rng = np.random.default_rng(seed + 7)
    n = len(X_tr)
    null = np.empty((B, len(idx)))
    ratios = []
    t0 = time.time()
    for b in range(B):
        order = _circular_block_permute(np.arange(n), block, rng).astype(int)
        Xb = X_tr.copy(); Xb[:, idx] = X_tr[order][:, idx]
        mb = make_model(model_name, seed=seed + 1 + b, **hp); mb.fit(Xb, y_tr)
        nb, svb = _explainer_mean_abs(mb, bg, X_eval, model_name)
        null[b] = nb[idx]
        if state_mask is not None and state_blocks:
            ratios.append(_state_ratio(svb, state_mask, state_blocks, feature_names))
        if verbose and (b + 1) % 50 == 0:
            print(f"cond-null {b+1}/{B}  ({time.time()-t0:.0f}s)", flush=True)

    o = obs[idx]
    p = (1 + (null >= o).sum(axis=0)) / (B + 1)
    tab = pd.DataFrame({"feature": test_cols, "mean|SHAP|": o,"cond-null mean": null.mean(axis=0),"cond-null q95": np.quantile(null, 0.95, axis=0), "ratio": o / np.maximum(null.mean(axis=0), 1e-30), "cond p": p,}).sort_values("mean|SHAP|", ascending=False).reset_index(drop=True)

    state_tab = None
    if ratios:
        obs_r = _state_ratio(sv_obs, state_mask, state_blocks, feature_names)
        R = pd.DataFrame(ratios)
        state_tab = pd.DataFrame([{ "block": b, "turb/calm": obs_r[b], "null mean": R[b].mean(),"null q95": R[b].quantile(0.95), "perm p": (1 + (R[b].values >= obs_r[b]).sum()) / (len(R) + 1),} for b in obs_r if b in R.columns])
    return tab, null, state_tab

def shap_linearity(sv, X_eval, feature_names, cols):
    rows = []
    for c in cols:
        j = feature_names.index(c)
        x = np.asarray(X_eval[:, j], float); phi = np.asarray(sv[:, j], float)
        ok = np.isfinite(x) & np.isfinite(phi)
        if ok.sum() < 50 or np.std(phi[ok]) < 1e-12:
            continue
        x, phi = x[ok], phi[ok]
        z = (x - x.mean()) / (x.std() + 1e-30)
        r1 = sm.OLS(phi, sm.add_constant(z)).fit()
        r3 = sm.OLS(phi, sm.add_constant(np.c_[z, z ** 2, z ** 3])).fit()
        rows.append({"feature": c, "R2 linear": r1.rsquared, "R2 cubic": r3.rsquared,"curvature": r3.rsquared - r1.rsquared, "slope": float(r1.params[1]) / (x.std() + 1e-30)})
    return pd.DataFrame(rows)

def shap_interaction_share(model, X_eval, model_name, feature_names):
    if model_name not in ("RandomForest", "ExtraTrees", "GBM", "XGB", "LGBM"):
        return None
    ex = shap.TreeExplainer(model)
    iv = ex.shap_interaction_values(X_eval)
    iv = np.asarray(iv)
    if iv.ndim == 4:
        iv = iv[..., 0]
    A = np.abs(iv).mean(axis=0)
    off = A.sum() - np.trace(A)
    total = A.sum()
    pairs = [(feature_names[i], feature_names[j], A[i, j]) for i in range(len(feature_names)) for j in range(i + 1, len(feature_names))]
    pairs.sort(key=lambda t: -t[2])
    return {"interaction_share": float(off / max(total, 1e-30)),"top_pairs": pairs[:10], "matrix": A}

def shap_by_state(sv, feat_eval_index, feat, blocks, feature_names):
    rv = feat["RV"].reindex(feat_eval_index)
    thr = feat["RV"].expanding(min_periods=100).quantile(2 / 3).shift(1).reindex(feat_eval_index)
    turb = (rv >= thr).values
    A = np.abs(sv)
    rows = []
    for bname, cols in blocks.items():
        idx = [feature_names.index(c) for c in cols if c in feature_names]
        if not idx:
            continue
        m = A[:, idx].sum(axis=1)
        rows.append({"block": bname, "calm": float(m[~turb].mean()), "turbulent": float(m[turb].mean()), "turb/calm": float(m[turb].mean() / max(m[~turb].mean(), 1e-30))})
    return pd.DataFrame(rows), turb

In [8]:
print("=" * 92); print("1. DATA"); print("=" * 92)
nifty_raw, vix_raw, G, missing = load_all()
rv = build_daily_measures(nifty_raw)
rv["VIX_close"] = vix_raw.groupby("date")["close"].last().reindex(rv.index)
rv["IV"] = (rv["VIX_close"] / 100) ** 2 / TRADING_DAYS_PER_YEAR
rv = rv.dropna(subset=["RV", "IV"]); rv = rv[rv["RV"] > 0]
print(f"\nDaily measures: {len(rv):,} days  {rv.index.min().date()} -> {rv.index.max().date()}")
print(f"\n annualised RV mean {100*np.sqrt(rv['RV'].mean()*TRADING_DAYS_PER_YEAR):.2f}%")
print("\n" + "=" * 92); print("2. LEAKAGE AUDIT"); print("=" * 92)
for nm in ("SPX", "VIX_US", "WTI", "FX"):
    if nm not in G:
        continue
    col = [c for c in G[nm].columns][0]
    a, lag = align_strictly_before(G[nm][col], rv.index, nm)
    assert_no_lookahead(G[nm][col], rv.index, a, nm)
    print(f"  [PASS] {nm:7} no look-ahead | coverage {100*a.notna().mean():5.1f}% " f"| lag days med {lag.median():.0f} max {lag.max():.0f}")
assert assert_folds_purged(1000, 5) and assert_folds_purged(1000, 22)
print("  [PASS] purged CV folds respect the h-day purge at h=5 and h=22")
feat, BLOCKS, notes = build_features(rv, G, missing, horizon=1)
SETS = feature_sets(BLOCKS)
print(f"\nFeature table {len(feat):,} rows  {feat.index.min().date()} -> {feat.index.max().date()}")
for k, v in BLOCKS.items():
    if v and k != "GLOBAL_ALL":
        print(f"  {k:12} {len(v):2d}  {v}")
for n_ in notes:
    print(f"  NOTE: {n_}")

print("\n" + "=" * 92); print("3. DATA-QUALITY SCAN on every feature"); print("=" * 92)
bad = []
for c in sorted({x for v in BLOCKS.values() for x in v}):
    s = feat[c].dropna()
    z = (s - s.mean()) / (s.std() + 1e-30)
    mx = float(z.abs().max())
    if mx > 12:
        one_sided = bool((s >= 0).all() or (s <= 0).all())
        bad.append((c, mx, float(s.abs().max()), one_sided, float(s.abs().max() / s.abs().quantile(0.999))))
if bad:
    print("columns with |z| > 12:")
    print(f"{'column':16} {'max|z|':>8} {'max|value|':>12} {'max/p99.9':>10}  verdict")
    for c, mx, amx, os_, ratio in sorted(bad, key=lambda t: -t[1]):
        v = ("one-sided by construction - expected" if os_ else ("REVIEW - extreme relative to p99.9" if ratio > 3 else "heavy-tailed but plausible"))
        print(f"{c:16} {mx:8.1f} {amx:12.4g} {ratio:10.2f}  {v}")
    print("a genuine artifact shows up as max/p99.9 >> 1 (the clipped negative WTI\n"
          "price scored 38x); a one-sided return column does not.")
else:
    print("no feature exceeds |z| = 12 - no obvious data artifacts")
assert_blocks_full_rank(feat, BLOCKS)
print("  [PASS] every block and their union are full rank - no exact linear dependence")

print("\n" + "=" * 92); print("4. RQ1  does the global channel exist (linear, HAC Wald)?"); print("=" * 92)
D = BLOCKS["DOMESTIC"]; y = feat["logRV_target"]; ok = y.notna()
r0 = sm.OLS(y[ok], sm.add_constant(feat[D])[ok]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
rows = []
for nm, extra in (("GLOBAL_EQ (S&P)", BLOCKS["GLOBAL_EQ"]), ("GLOBAL_IV (VIX)", BLOCKS["GLOBAL_IV"]), ("GLOBAL_FX", BLOCKS["GLOBAL_FX"]), ("GLOBAL_CMD", BLOCKS["GLOBAL_CMD"]), ("ALL GLOBAL", BLOCKS["GLOBAL_ALL"]), ("ALL + interactions", BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"])):
    if not extra:
        continue
    X1 = sm.add_constant(feat[D + extra])
    r1 = sm.OLS(y[ok], X1[ok]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
    R = np.zeros((len(extra), X1.shape[1]))
    for j, c in enumerate(extra):
        R[j, list(X1.columns).index(c)] = 1.0
    w = r1.f_test(R)
    rows.append({"block": nm, "k": len(extra), "dR2 x100": 100 * (r1.rsquared - r0.rsquared), "Wald F": float(w.fvalue), "p": float(w.pvalue)})
print(f"base R2 = {r0.rsquared:.4f}\n")
rq1_tab = pd.DataFrame(rows)          # stable name: later cells reuse `rows`
print(rq1_tab.to_string(index=False, float_format=lambda x: f"{x:10.4f}"))

print("\nSingle-regressor tests (each added ALONE to the domestic benchmark):")
solo = []
for c in BLOCKS["GLOBAL_ALL"]:
    X1 = sm.add_constant(feat[D + [c]])
    r1 = sm.OLS(y[ok], X1[ok]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
    solo.append({"feature": c, "coef": r1.params[c], "HAC t": r1.tvalues[c], "p": r1.pvalues[c], "dR2 x100": 100 * (r1.rsquared - r0.rsquared)})
solo_tab = pd.DataFrame(solo).sort_values("p")
print(solo_tab.to_string(index=False, float_format=lambda x: f"{x:10.4f}"))

print("\nBenjamini-Hochberg over the single-regressor scan:")
_sc = pd.DataFrame(solo)
_sig = benjamini_hochberg(_sc["p"].values, list(_sc["feature"]), alpha=0.10)
print(f"  {len(_sc)} tests, FDR 10% -> {sorted(_sig) if _sig else 'NOTHING SURVIVES'}")
print(f"  raw p<0.05: {(_sc['p'] < 0.05).sum()} of {len(_sc)} " f"(expected under the null: {0.05*len(_sc):.1f})")

_X1 = sm.add_constant(feat[D + BLOCKS["DOM_EXT"]])
_r1 = sm.OLS(y[ok], _X1[ok]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
_R = np.zeros((len(BLOCKS["DOM_EXT"]), _X1.shape[1]))
for _j, _c in enumerate(BLOCKS["DOM_EXT"]):
    _R[_j, list(_X1.columns).index(_c)] = 1.0
_w = _r1.f_test(_R)
print(f"\nPOWER CHECK - the purely DOMESTIC extension block on the same benchmark:")
print(f"  DOM_EXT (k={len(BLOCKS['DOM_EXT'])}): dR2x100 {100*(_r1.rsquared-r0.rsquared):.4f}" f"  F {float(_w.fvalue):.3f}  p {float(_w.pvalue):.4f}")

domext_dr2 = 100 * (_r1.rsquared - r0.rsquared)
domext_p   = float(_w.pvalue)
solo_sig   = sorted(_sig) if _sig else []
solo_n     = len(_sc)

1. DATA
Reading all inputs from ./Datasets/   (no network access)

  [PASS] both conventions parsed with no day/month transposition
  NIFTY PSU BANK_5minute.csv: 209,637 bars kept | 0 unparsed, 120 out-of-session, 23 stub-day bars dropped | 2,797 days [ISO/month-first]
  INDIA VIX_5minute.csv: 209,571 bars kept | 0 unparsed, 140 out-of-session, 23 stub-day bars dropped | 2,796 days [day-first]
  [SPX] 3,037 rows  2014-06-02 -> 2026-06-29 [ISO/month-first]
  [VIX_US] 3,037 rows  2014-06-02 -> 2026-06-29 [ISO/month-first]
  [WTI] 4,154 rows  2010-01-04 -> 2026-07-27   (column 'price' -> 'price') [day-first]
  [FX] 13,974 rows  1973-01-02 -> 2026-07-24   (column 'exchange rate' -> 'usdinr') [day-first]

Daily measures: 2,795 days  2015-01-12 -> 2026-05-18

 annualised RV mean 26.42%

2. LEAKAGE AUDIT
  [PASS] SPX     no look-ahead | coverage 100.0% | lag days med 1 max 4
  [PASS] VIX_US  no look-ahead | coverage 100.0% | lag days med 1 max 4
  [PASS] WTI     no look-ahead | coverage 100.0

In [9]:
# Benchmark Specifications
BENCH_SPECS = {
    "HAR-RV (levels)":(["RV_1", "RV_5", "RV_22"], "levels"),
    "log-HAR-RV": (["logRV_1", "logRV_5", "logRV_22"], "log"),
    "log-HAR-RV-IV": (["logRV_1", "logRV_5", "logRV_22", "logIV_1"], "log"),
    "log-HAR-RV-IV-LFULL": (["logRV_1", "logRV_5", "logRV_22", "logIV_1", "r_pos_1", "r_neg_1", "r_pos_5", "r_neg_5", "r_pos_22", "r_neg_22"], "log"),
    "log-HAR-RV-IV-LFULL-VoV": (["logRV_1", "logRV_5", "logRV_22", "logIV_1", "r_pos_1", "r_neg_1", "r_pos_5", "r_neg_5", "r_pos_22", "r_neg_22", "logVoV_22"], "log"),
}

ML_BENCHMARKS = ["log-HAR-RV-IV", "log-HAR-RV-IV-LFULL", "log-HAR-RV-IV-LFULL-VoV"]

def run_bench_ladder(feat, h=1, oos_fraction=OOS_FRACTION, verbose=False):
    usable = feat[feat["RV_target"].notna()]
    n0 = int(len(usable) * (1 - oos_fraction))
    oos = usable.index[n0:]
    idx_of = {d: i for i, d in enumerate(feat.index)}
    preds = {k: [] for k in BENCH_SPECS}
    actuals, origins = [], []
    t0 = time.time()
    for i, t in enumerate(oos):
        pos = idx_of[t]; end = pos - h + 1
        if end < 300:
            continue
        tr = feat.iloc[:end]; tr = tr[tr["RV_target"].notna()]
        if len(tr) < 300:
            continue
        y_true = feat["RV_target"].iloc[pos]
        if not np.isfinite(y_true) or y_true <= 0:
            continue
        actuals.append(y_true); origins.append(t)
        floor = max(PRED_FLOOR_FRAC * float(tr["RV"].median()), 1e-16)
        for nm, (cols, kind) in BENCH_SPECS.items():
            X = sm.add_constant(tr[cols].values)
            xn = np.r_[1.0, feat[cols].iloc[pos].values].reshape(1, -1)
            if kind == "levels":
                r = sm.OLS(tr["RV_target"].values, X).fit()
                preds[nm].append(max(float(r.predict(xn)[0]), floor))
            else:
                r = sm.OLS(tr["logRV_target"].values, X).fit()
                mu = float(r.predict(xn)[0])
                preds[nm].append(max(np.exp(mu + 0.5 * float(np.var(r.resid))), floor))
        if verbose and i % 200 == 0 and i > 0:
            print(f"ladder origin {i}/{len(oos)} ({time.time()-t0:.0f}s)", flush=True)
    out = pd.DataFrame(preds, index=pd.DatetimeIndex(origins))
    out["actual"] = actuals
    return out

def bench_table(fc_bench):
    a = fc_bench["actual"].values
    ref = "log-HAR-RV-IV-LFULL"
    qref = qlike(a, fc_bench[ref].values)
    rows = []
    for nm in BENCH_SPECS:
        q = qlike(a, fc_bench[nm].values)
        t, p = (np.nan, np.nan) if nm == ref else dm(qref, q)
        rows.append({"benchmark": nm, "k": len(BENCH_SPECS[nm][0]),"QLIKE": q.mean(),"vs selected %": 100 * (q.mean() / qref.mean() - 1),"DM t": t, "DM p": p})                     
    return pd.DataFrame(rows).sort_values("QLIKE").reset_index(drop=True)

In [10]:
fc_bench = run_bench_ladder(feat, h=PRIMARY_HORIZON, verbose=False)
print(f"Benchmark ladder, walk-forward OLS, {len(fc_bench):,} origins")
print("'vs selected' is measured against log-HAR-RV-IV-LFULL, Part I's chosen model\n")
print(bench_table(fc_bench).to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

Benchmark ladder, walk-forward OLS, 802 origins
'vs selected' is measured against log-HAR-RV-IV-LFULL, Part I's chosen model

              benchmark  k     QLIKE  vs selected %      DM t      DM p
    log-HAR-RV-IV-LFULL 10    0.1815         0.0000       NaN       NaN
log-HAR-RV-IV-LFULL-VoV 11    0.1826         0.6334   -1.0831    0.2791
          log-HAR-RV-IV  4    0.1931         6.3793   -1.3846    0.1665
             log-HAR-RV  3    0.1983         9.2390   -1.6253    0.1045
        HAR-RV (levels)  3    0.2763        52.2635   -6.1001    0.0000


In [11]:
MX_MODELS = ["Ridge", "ElasticNet", "RandomForest", "GBM", "XGB", "MLP"]
MX_SEEDS  = (0, 1, 2)
GB = BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"]

mx_fc, mx_summary = {}, []
for _bn in ML_BENCHMARKS:
    _bcols = BENCH_SPECS[_bn][0]
    for _fn, _fcols in (("same", _bcols), ("global", GB)):
        mx_fc[(_bn, _fn)] = run_har_boost(feat, _fcols, _bcols, MX_MODELS, h=PRIMARY_HORIZON, seeds=MX_SEEDS, verbose=False)
                                         
for _bn in ML_BENCHMARKS:
    for _fn in ("same", "global"):
        _fc = mx_fc[(_bn, _fn)]
        _a = _fc["actual"].values
        _qb = qlike(_a, _fc["HAR_bench"].values)
        _rows = []
        for _m in MX_MODELS:
            _cols = [c for c in _fc.columns if c.startswith(f"BOOST_{_m}|")]
            _sa = np.exp(np.log(_fc[_cols].values).mean(axis=1))
            _q = qlike(_a, _sa); _t, _p = dm(_qb, _q)
            _adj = np.log(_sa) - np.log(_fc["HAR_bench"].values)
            _rows.append({"model": _m, "QLIKE": _q.mean(),"vs bench %": 100*(_q.mean()/_qb.mean()-1),"DM t": _t, "DM p": _p, "mean |adj|": np.abs(_adj).mean()})
        _tb = pd.DataFrame(_rows).sort_values("QLIKE").reset_index(drop=True)
        _sig = benjamini_hochberg(_tb["DM p"].values, list(_tb["model"]), alpha=0.10)
        _bet = [m for m in _sig if _tb.set_index("model").loc[m, "vs bench %"] < 0]
        _k = len(BENCH_SPECS[_bn][0]) if _fn == "same" else len(GB)
        print(f"\n=== benchmark {_bn}   |   ML features: {_fn.upper()} (k={_k}) ===")
        print(f"    benchmark QLIKE = {_qb.mean():.4f}   ({len(_fc):,} origins)")
        print(_tb.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
        print(f"    BH-FDR 10% BETTER than this benchmark: {_bet if _bet else 'NONE'}")
        mx_summary.append({"benchmark": _bn, "ML features": _fn,"bench QLIKE": _qb.mean(), "best ML": _tb.iloc[0]["model"],"best QLIKE": _tb.iloc[0]["QLIKE"],"best vs bench %": _tb.iloc[0]["vs bench %"], "best DM p": _tb.iloc[0]["DM p"],"n better @FDR10": len(_bet)})

print("\n\n=== SUMMARY MATRIX  ===\n")
mx_tab = pd.DataFrame(mx_summary)
print(mx_tab.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
print(f"\ncells in which ANY model beats its benchmark at FDR 10%: "
      f"{int((mx_tab['n better @FDR10'] > 0).sum())} of {len(mx_tab)}")


=== benchmark log-HAR-RV-IV   |   ML features: SAME (k=4) ===
    benchmark QLIKE = 0.1931   (802 origins)
       model     QLIKE  vs bench %      DM t      DM p  mean |adj|
       Ridge    0.1931     -0.0000    0.0000    1.0000      0.0000
  ElasticNet    0.1931     -0.0000    0.0000    1.0000      0.0000
RandomForest    0.1954      1.2263   -1.7810    0.0753      0.0206
         MLP    0.1957      1.3551   -2.1446    0.0323      0.0316
         GBM    0.1959      1.4857   -1.2164    0.2242      0.0461
         XGB    0.2006      3.8818   -1.6897    0.0915      0.0555
    BH-FDR 10% BETTER than this benchmark: NONE

=== benchmark log-HAR-RV-IV   |   ML features: GLOBAL (k=15) ===
    benchmark QLIKE = 0.1931   (802 origins)
       model     QLIKE  vs bench %      DM t      DM p  mean |adj|
  ElasticNet    0.1925     -0.2864    0.7939    0.4275      0.0141
       Ridge    0.1926     -0.2646    0.3394    0.7344      0.0282
RandomForest    0.1935      0.2476   -0.3080    0.7581      0.0

In [12]:
# Comparing with sequence models
import torch
import torch.nn as nn

SEQ_LEN          = 22      
SEQ_HIDDEN       = 16  
SEQ_EPOCHS       = 120    
SEQ_PATIENCE     = 12     
SEQ_BATCH        = 64
SEQ_LR           = 3e-3
SEQ_WD           = 1e-4   
SEQ_VAL_FRAC     = 0.15   
SEQ_REFIT_EVERY  = 63      
torch.set_num_threads(2)

class _SeqNet(nn.Module):
    def __init__(self, n_features, kind="LSTM", hidden=SEQ_HIDDEN, dropout=0.1):
        super().__init__()
        rnn = nn.LSTM if kind == "LSTM" else nn.GRU
        self.rnn = rnn(n_features, hidden, num_layers=1, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)

def make_sequences(X, y, seq_len=SEQ_LEN):
    n, k = X.shape
    if n <= seq_len:
        return np.empty((0, seq_len, k), dtype=np.float32), np.empty((0,), dtype=np.float32), []
    idx = np.arange(seq_len - 1, n)
    S = np.stack([X[i - seq_len + 1: i + 1] for i in idx]).astype(np.float32)
    return S, np.asarray(y, dtype=np.float32)[idx], list(idx)

def fit_seq_model(X_tr, y_tr, kind="LSTM", seed=0, h=1, seq_len=SEQ_LEN, hidden=SEQ_HIDDEN, epochs=SEQ_EPOCHS, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    n = len(X_tr)
    n_val = max(seq_len + 20, int(SEQ_VAL_FRAC * n))
    cut = n - n_val
    purge = h + seq_len
    tr_end = max(seq_len + 10, cut - purge)

    mu = X_tr[:tr_end].mean(axis=0); sd = X_tr[:tr_end].std(axis=0); sd[sd == 0] = 1.0
    ys = float(np.std(y_tr[:tr_end])) or 1.0
    Z = (X_tr - mu) / sd
    yz = y_tr / ys

    S_tr, t_tr, _ = make_sequences(Z[:tr_end], yz[:tr_end], seq_len)
    S_va, t_va, _ = make_sequences(Z[cut - seq_len + 1:], yz[cut - seq_len + 1:], seq_len)
    if len(S_tr) < 50 or len(S_va) < 10:
        return None
    m = _SeqNet(X_tr.shape[1], kind=kind, hidden=hidden)
    opt = torch.optim.Adam(m.parameters(), lr=SEQ_LR, weight_decay=SEQ_WD)
    lossf = nn.MSELoss()
    Xt, yt = torch.from_numpy(S_tr), torch.from_numpy(t_tr)
    Xv, yv = torch.from_numpy(S_va), torch.from_numpy(t_va)
    best, best_state, bad = np.inf, None, 0
    nb = int(np.ceil(len(Xt) / SEQ_BATCH))
    g = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xt), generator=g)
        for b in range(nb):
            sl = perm[b * SEQ_BATCH:(b + 1) * SEQ_BATCH]
            opt.zero_grad(); loss = lossf(m(Xt[sl]), yt[sl]); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        m.eval()
        with torch.no_grad():
            v = float(lossf(m(Xv), yv))
        if v < best - 1e-6:
            best, bad = v, 0
            best_state = {k: t.detach().clone() for k, t in m.state_dict().items()}
        else:
            bad += 1
            if bad >= SEQ_PATIENCE:
                break
    if best_state is not None:
        m.load_state_dict(best_state)
    m.eval()
    return {"model": m, "mu": mu, "sd": sd, "ys": ys, "seq_len": seq_len, "epochs_used": ep + 1, "val_mse": best}


def seq_predict_last(state, X_hist):
    if state is None:
        return 0.0
    L = state["seq_len"]
    if len(X_hist) < L:
        return 0.0
    Z = (X_hist[-L:] - state["mu"]) / state["sd"]
    with torch.no_grad():
        v = float(state["model"](torch.from_numpy(Z[None, :, :].astype(np.float32)))[0])
    return v * state["ys"]

def run_har_boost_seq(feat, global_cols, bench_cols, kinds=("LSTM", "GRU"), h=1,  seeds=(0, 1, 2), oos_fraction=OOS_FRACTION,  refit_every=SEQ_REFIT_EVERY, seq_len=SEQ_LEN, verbose=False):
    usable = feat[feat["RV_target"].notna()]
    n0 = int(len(usable) * (1 - oos_fraction))
    oos = usable.index[n0:]
    Xg_all = feat[global_cols].values.astype(float)
    idx_of = {d: i for i, d in enumerate(feat.index)}

    preds = {f"BOOST_{k}|s{s}": [] for k in kinds for s in seeds}
    preds["HAR_bench"] = []
    actuals, origins, state = [], [], {}
    t0 = time.time()
    for i, t in enumerate(oos):
        pos = idx_of[t]; end = pos - h + 1
        if end < 300:
            continue
        tr = feat.iloc[:end]; tr = tr[tr["RV_target"].notna()]
        if len(tr) < 300:
            continue
        y_true = feat["RV_target"].iloc[pos]
        if not np.isfinite(y_true) or y_true <= 0:
            continue
        actuals.append(y_true); origins.append(t)
        ytr = tr["logRV_target"].values.astype(float)
        floor = max(PRED_FLOOR_FRAC * float(tr["RV"].median()), 1e-16)

        Xb = sm.add_constant(tr[bench_cols].values)
        rb = sm.OLS(ytr, Xb).fit()
        sig2 = float(np.var(rb.resid))
        mub = float(rb.predict(np.r_[1.0, feat[bench_cols].iloc[pos].values].reshape(1, -1))[0])
        preds["HAR_bench"].append(max(np.exp(mub + 0.5 * sig2), floor))

        e_tr = ytr - rb.fittedvalues
        Xg_tr = feat.loc[tr.index, global_cols].values.astype(float)
        for kind in kinds:
            for s in seeds:
                key = (kind, s)
                if (i % refit_every == 0) or (key not in state):
                    state[key] = fit_seq_model(Xg_tr, e_tr, kind=kind, seed=s, h=h,seq_len=seq_len)                     
                adj = seq_predict_last(state[key], Xg_all[:pos + 1])
                preds[f"BOOST_{kind}|s{s}"].append(max(np.exp(mub + adj + 0.5 * sig2), floor))
        if verbose and i % 100 == 0 and i > 0:
            print(f"seq origin {i}/{len(oos)} ({time.time()-t0:.0f}s)", flush=True)

    out = pd.DataFrame({k: v for k, v in preds.items() if len(v) == len(actuals)},index=pd.DatetimeIndex(origins))                      
    out["actual"] = actuals
    return out

def assert_seq_no_lookahead(feat, global_cols, seq_len=SEQ_LEN):
    X = feat[global_cols].values.astype(float)
    p = len(X) // 2
    a = X[p - seq_len + 1: p + 1].copy()
    Xb = X.copy(); Xb[p + 1:] = -98765.0
    b = Xb[p - seq_len + 1: p + 1]
    assert np.array_equal(a, b), "sequence window leaks future rows"
    return True

In [13]:
IDENT_TOL = 1e-3

def seed_average(fc, family, seeds=(0, 1, 2)):   
    cols = [f"BOOST_{family}|s{s}" for s in seeds if f"BOOST_{family}|s{s}" in fc]
    if not cols:
        return None
    return np.exp(np.log(fc[cols].values).mean(axis=1))


def combine_equal(fc, members):
    """Equal-weight combination in logs (Bates & Granger 1969, simple case)."""
    return np.exp(np.log(fc[members].values).mean(axis=1))

def combine_inverse_loss(fc, members, actual, warmup=120, h=1, return_weights=False):
    A = np.asarray(actual, float)
    L = np.column_stack([qlike(A, fc[m].values) for m in members])
    logF = np.log(fc[members].values)
    n, k = L.shape
    out = np.empty(n)
    W = np.empty((n, k))
    for t in range(n):
        if t < warmup:
            w = np.full(k, 1.0 / k)
        else:
            mean_past = L[:t].mean(axis=0)
            inv = 1.0 / np.maximum(mean_past, 1e-12)
            w = inv / inv.sum()
        W[t] = w
        out[t] = float(logF[t] @ w)
    return (np.exp(out), W) if return_weights else np.exp(out)


def member_dispersion(fc, members):
    return float(np.log(fc[members].values).std(axis=1, ddof=0).mean())


def assert_combination_causal(fc, members, actual, warmup=120, verbose=False):
    a = np.asarray(actual, float).copy()
    n = len(a)
    k = n // 2
    base, W_base = combine_inverse_loss(fc, members, a, warmup=warmup, return_weights=True)
    a_bad = a.copy()
    a_bad[k:] = np.asarray(fc[members[0]].values, float)[k:]
    bad, W_bad = combine_inverse_loss(fc, members, a_bad, warmup=warmup, return_weights=True)
    assert np.allclose(W_base[:k + 1], W_bad[:k + 1]), \
        "inverse-loss combination LEAKS: future actuals changed an earlier WEIGHT"
    assert np.allclose(base[:k + 1], bad[:k + 1]), \
        "inverse-loss combination LEAKS: future actuals changed an earlier forecast"

    disp = member_dispersion(fc, members)
    identified = disp >= IDENT_TOL

    post = max(k + 1, warmup)
    moved = float("nan")
    if identified and n > post + 1:
        assert not np.allclose(W_base[post:], W_bad[post:]), \
            "inverse-loss weights are not responding to the loss history"
        moved = float(np.abs(W_base[post:] - W_bad[post:]).max())

    if verbose:
        if identified:
            print(f"    causality OK | max weight shift {moved:.4f} | "
                  f"member dispersion {disp:.5f}")
        else:
            print(f"    causality OK | member dispersion {disp:.2e} < {IDENT_TOL:.0e} "
                  f"-> weights NOT IDENTIFIED, responsiveness not testable.\n"
                  f"      The ML members have collapsed onto the HAR benchmark for this\n"
                  f"      index, so every weighting rule returns the same series. Report\n"
                  f"      the combination as degenerate rather than as a result.\n"
                  f"      NOTE: in this regime the audit certifies nothing. With identical\n"
                  f"      members a LEAKY full-sample weight returns the same numbers as the\n"
                  f"      causal recursive one, so leakage is undetectable here - and equally,\n"
                  f"      inconsequential, since every weighting coincides. The causality\n"
                  f"      guarantee for this study rests on the indices where disp >= tol.")
    return {"causal": True, "identified": identified,
            "dispersion": disp, "max_weight_shift": moved}

In [14]:
# Results
GB = BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"]
STATIC = ["Ridge", "ElasticNet", "RandomForest", "GBM", "XGB", "MLP"]
SEEDS  = (0, 1, 2)

fc_static = run_har_boost(feat, GB, BLOCKS["DOMESTIC"], STATIC,  h=PRIMARY_HORIZON, seeds=SEEDS, verbose=False)
assert assert_seq_no_lookahead(feat, GB)
fc_seq = run_har_boost_seq(feat, GB, BLOCKS["DOMESTIC"], kinds=("LSTM", "GRU"),  h=PRIMARY_HORIZON, seeds=SEEDS, verbose=False)
assert fc_static.index.equals(fc_seq.index)
fc_boost = fc_static.join(fc_seq.drop(columns=["actual", "HAR_bench"]))
a = fc_boost["actual"].values
q_har = qlike(a, fc_boost["HAR_bench"].values)
FAMS = STATIC + ["LSTM", "GRU"]
for _f in FAMS:
    fc_boost[f"SEEDAVG_{_f}"] = seed_average(fc_boost, _f, SEEDS)
SAVG = [f"SEEDAVG_{f}" for f in FAMS]
fc_boost["COMB_equal_ML"]  = combine_equal(fc_boost, SAVG)
fc_boost["COMB_equal_HAR_ML"] = combine_equal(fc_boost, ["HAR_bench"] + SAVG)
assert assert_combination_causal(fc_boost, ["HAR_bench"] + SAVG, a)
fc_boost["COMB_invloss"] = combine_inverse_loss(fc_boost, ["HAR_bench"] + SAVG, a)

rows = []
for c in fc_boost.columns:
    if c == "actual":
        continue
    ql = qlike(a, fc_boost[c].values)
    t, p = (np.nan, np.nan) if c == "HAR_bench" else dm(q_har, ql)
    rows.append({"model": c, "QLIKE": ql.mean(), "vs HAR %": 100*(ql.mean()/q_har.mean()-1), "DM t": t, "DM p": p})
boost_tab = pd.DataFrame(rows).sort_values("QLIKE").reset_index(drop=True)
print(f"HAR benchmark QLIKE = {q_har.mean():.4f}   ({len(fc_boost):,} origins, "
      f"{len(FAMS)} model families, {len(SEEDS)} seeds)\n")
print(boost_tab.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

_p  = boost_tab.dropna(subset=["DM p"])
_s  = benjamini_hochberg(_p["DM p"].values, list(_p["model"]), alpha=0.10)
_bt = [m for m in _s if boost_tab.set_index("model").loc[m, "vs HAR %"] < 0]
_ws = [m for m in _s if boost_tab.set_index("model").loc[m, "vs HAR %"] > 0]
print(f"\nBH-FDR 10%   BETTER than HAR : {_bt if _bt else 'NONE'}")
print(f"BH-FDR 10%   WORSE  than HAR : {len(_ws)} of {len(_p)} candidates")

print("\n\nSEED DISPERSION, AND WHAT AVERAGING OVER SEEDS RECOVERS\n")
print(f"  {'family':14} {'seed range':>22} {'sd':>9} {'seed-avg':>10} {'gain':>9}")
for _f in FAMS:
    _cs = [f"BOOST_{_f}|s{s}" for s in SEEDS]
    _qs = [qlike(a, fc_boost[c].values).mean() for c in _cs]
    _sa = qlike(a, fc_boost[f"SEEDAVG_{_f}"].values).mean()
    print(f"  {_f:14} [{min(_qs):.4f}, {max(_qs):.4f}] {np.std(_qs):9.5f}"
          f" {_sa:10.4f} {np.mean(_qs)-_sa:+9.5f}")
_nonhar = boost_tab[boost_tab["model"] != "HAR_bench"]
_gap = abs(_nonhar.iloc[0]["QLIKE"] - q_har.mean())
print(f"\n best NON-HAR candidate: {_nonhar.iloc[0]['model']}"
      f"its gap to HAR = {_gap:.5f}")
print(f"largest seed sd among the stochastic families = "
      f"{max(np.std([qlike(a, fc_boost[f'BOOST_{f}|s{s}'].values).mean() for s in SEEDS]) for f in FAMS):.5f}")

print("\n\nMODEL CONFIDENCE SET over HAR + seed-averaged models + combinations\n")
_C = ["HAR_bench"] + SAVG + ["COMB_equal_ML", "COMB_equal_HAR_ML", "COMB_invloss"]
_L = pd.DataFrame({m: qlike(a, fc_boost[m].values) for m in _C})
mcs_surv, mcs_elim = model_confidence_set(_L, alpha=0.10, n_boot=1000, block=10)
LED_mcs = (len(mcs_surv), len(_C), "HAR_bench" in mcs_surv)
print(f"  superior set ({len(mcs_surv)} of {len(_C)}): {sorted(mcs_surv)}")
print(f"  eliminated  : {[m for m, _ in mcs_elim]}")

HAR benchmark QLIKE = 0.1815   (802 origins, 8 model families, 3 seeds)

                model     QLIKE  vs HAR %      DM t      DM p
  BOOST_ElasticNet|s0    0.1814   -0.0438    0.1951    0.8454
  BOOST_ElasticNet|s1    0.1814   -0.0438    0.1951    0.8454
  BOOST_ElasticNet|s2    0.1814   -0.0438    0.1951    0.8454
   SEEDAVG_ElasticNet    0.1814   -0.0438    0.1951    0.8454
            HAR_bench    0.1815    0.0000       NaN       NaN
         BOOST_GRU|s2    0.1818    0.1877   -0.3192    0.7496
       BOOST_Ridge|s2    0.1819    0.2040   -0.4119    0.6805
       BOOST_Ridge|s0    0.1819    0.2040   -0.4119    0.6805
        SEEDAVG_Ridge    0.1819    0.2040   -0.4119    0.6805
       BOOST_Ridge|s1    0.1819    0.2040   -0.4119    0.6805
BOOST_RandomForest|s2    0.1820    0.2831   -0.4065    0.6845
BOOST_RandomForest|s0    0.1821    0.3616   -0.5519    0.5812
 SEEDAVG_RandomForest    0.1822    0.3657   -0.5333    0.5939
BOOST_RandomForest|s1    0.1824    0.4830   -0.6716    0.50

In [15]:

_lh = np.log(fc_boost["HAR_bench"].values)
_rows = []
for _f in FAMS:
    _c = f"SEEDAVG_{_f}"
    _adj = np.log(fc_boost[_c].values) - _lh
    _rows.append({"model": _f, "QLIKE": qlike(a, fc_boost[_c].values).mean(),"vs HAR %": 100*(qlike(a, fc_boost[_c].values).mean()/q_har.mean()-1),"mean |adj|": np.abs(_adj).mean(), "sd adj": _adj.std(),"max |adj|": np.abs(_adj).max()})             
_at = pd.DataFrame(_rows).sort_values("QLIKE").reset_index(drop=True)
print("adj = log(ML forecast) - log(HAR forecast), in log-variance units\n")
print(_at.to_string(index=False, float_format=lambda x: f"{x:10.5f}"))
_rho = _at["mean |adj|"].corr(_at["vs HAR %"], method="spearman")
_r   = _at["mean |adj|"].corr(_at["vs HAR %"])
print(f"\n  Pearson  corr( mean |adjustment| , QLIKE penalty ) = {_r:+.3f}")
print(f"  Spearman rank correlation = {_rho:+.3f}")
LED_rho = float(_rho)

adj = log(ML forecast) - log(HAR forecast), in log-variance units

       model      QLIKE   vs HAR %  mean |adj|     sd adj  max |adj|
  ElasticNet    0.18141   -0.04379     0.01119    0.01530    0.09904
       Ridge    0.18186    0.20403     0.02247    0.03110    0.22437
RandomForest    0.18215    0.36570     0.02608    0.03796    0.16084
         GRU    0.18238    0.49058     0.02252    0.02893    0.10799
        LSTM    0.18288    0.76482     0.02332    0.03049    0.24659
         GBM    0.18500    1.93349     0.04823    0.06959    0.37090
         MLP    0.18587    2.41457     0.04872    0.06810    0.47798
         XGB    0.18640    2.70779     0.06009    0.08115    0.31781

  Pearson  corr( mean |adjustment| , QLIKE penalty ) = +0.975
  Spearman rank correlation = +0.929


In [16]:
# Explaining results 
DOM, DEX = BLOCKS["DOMESTIC"], BLOCKS["DOM_EXT"]
fc_a, _ = run_walkforward(feat, DOM, STATIC, DOM, h=PRIMARY_HORIZON,  seeds=SEEDS, verbose=False)
fc_b = run_har_boost(feat, DOM, DOM, STATIC, h=PRIMARY_HORIZON, seeds=SEEDS, verbose=False)
fc_c = run_har_boost(feat, DEX, DOM, STATIC, h=PRIMARY_HORIZON, seeds=SEEDS, verbose=False)

_a2 = fc_a["actual"].values
_qh2 = qlike(_a2, fc_a["HAR_bench"].values)
print(f"HAR benchmark QLIKE = {_qh2.mean():.4f}   ({len(fc_a):,} origins)\n")

def _same_table(fc, label, prefix=""):
    _rows = []
    for _f in STATIC:
        _cols = [c for c in fc.columns if c.startswith(f"{prefix}{_f}|")]
        if not _cols:
            continue
        _sa = np.exp(np.log(fc[_cols].values).mean(axis=1))
        _q = qlike(_a2, _sa); _t, _p = dm(_qh2, _q)
        _qs = [qlike(_a2, fc[c].values).mean() for c in _cols]
        _adj = np.log(_sa) - np.log(fc["HAR_bench"].values)
        _rows.append({"model": _f, "QLIKE": _q.mean(),  "vs HAR %": 100*(_q.mean()/_qh2.mean()-1),  "DM t": _t, "DM p": _p, "seed sd": np.std(_qs),  "mean |adj|": np.abs(_adj).mean()})
    _t2 = pd.DataFrame(_rows).sort_values("QLIKE").reset_index(drop=True)
    print(f"--- {label} ---")
    print(_t2.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
    _sig = benjamini_hochberg(_t2["DM p"].values, list(_t2["model"]), alpha=0.10)
    _bet = [m for m in _sig if _t2.set_index("model").loc[m, "vs HAR %"] < 0]
    print(f"BH-FDR 10%  BETTER than HAR: {_bet if _bet else 'NONE'}\n")
    return _t2
tab_a = _same_table(fc_a, "(a) ML REPLACES HAR - direct, the same 10 domestic features")
tab_b = _same_table(fc_b, "(b) ML on the HAR RESIDUAL - the same 10 domestic features", "BOOST_")
tab_c = _same_table(fc_c, "(c) POSITIVE CONTROL - ML on the HAR residual, 7 extra DOMESTIC ""features that OLS can exploit (dR2 0.41%, p=0.0059)", "BOOST_")
_lin = {"HAR": [], "HAR+DOM_EXT": []}; _acts = []
_usable = feat[feat["RV_target"].notna()]
_n0d = int(len(_usable) * (1 - OOS_FRACTION))
_oosd = _usable.index[_n0d:]
_idxd = {d: i for i, d in enumerate(feat.index)}
for _t in _oosd:
    _pos = _idxd[_t]; _tr = feat.iloc[:_pos]
    _tr = _tr[_tr["RV_target"].notna()]
    if len(_tr) < 300:
        continue
    _yv = feat["RV_target"].iloc[_pos]
    if not np.isfinite(_yv) or _yv <= 0:
        continue
    _acts.append(_yv); _ytr = _tr["logRV_target"].values
    _fl = max(PRED_FLOOR_FRAC * float(_tr["RV"].median()), 1e-16)
    for _nm, _cs in (("HAR", DOM), ("HAR+DOM_EXT", DOM + DEX)):
        _X = sm.add_constant(_tr[_cs].values); _r = sm.OLS(_ytr, _X).fit()
        _mu = float(_r.predict(np.r_[1.0, feat[_cs].iloc[_pos].values].reshape(1, -1))[0])
        _lin[_nm].append(max(np.exp(_mu + 0.5 * float(np.var(_r.resid))), _fl))
_fl2 = pd.DataFrame(_lin, index=pd.DatetimeIndex(_oosd[-len(_acts):]))
_ad = np.asarray(_acts)
_qhd = qlike(_ad, _fl2["HAR"].values); _qxd = qlike(_ad, _fl2["HAR+DOM_EXT"].values)
_td, _pd = dm(_qhd, _qxd)
print("\n--- (d) can a LINEAR model exploit DOM_EXT out of sample? (OLS walk-forward) ---\n")
print(f"HAR {_qhd.mean():.4f}")
print(f"HAR + DOM_EXT  {_qxd.mean():.4f}   ({100*(_qxd.mean()/_qhd.mean()-1):+.2f}%, "
      f"DM t {_td:+.2f}, p {_pd:.4f})")
LED_ols_domext = (100*(_qxd.mean()/_qhd.mean()-1), float(_pd))
print(" compare with design (c): if OLS also fails, the binding constraint is")
print("estimation noise at this effect size, not the ML machinery.\n")

print("\nREGIME SPLIT for each design (ex-ante state)\n")
_thr3 = feat["RV"].expanding(min_periods=100).quantile(2/3).shift(1).reindex(fc_a.index)
_tb3 = (feat["RV"].reindex(fc_a.index) >= _thr3).values
for _lbl, _fc, _pre in (("(a) direct", fc_a, ""), ("(b) resid same", fc_b, "BOOST_"),("(c) resid DOM_EXT", fc_c, "BOOST_")):                        
    for _st, _m in (("calm", ~_tb3), ("turb", _tb3)):
        _qb = qlike(_a2[_m], _fc["HAR_bench"].values[_m])
        _best, _bq, _bp = None, np.inf, np.nan
        for _f in STATIC:
            _cols = [c for c in _fc.columns if c.startswith(f"{_pre}{_f}|")]
            _sa = np.exp(np.log(_fc[_cols].values).mean(axis=1))
            _q = qlike(_a2[_m], _sa[_m])
            if _q.mean() < _bq:
                _best, _bq = _f, _q.mean(); _bp = dm(_qb, _q)[1]
        print(f"  {_lbl:18} {_st:5} n={int(_m.sum()):4d}  HAR {_qb.mean():.4f}"
              f"  best {_best:13} {_bq:.4f} ({100*(_bq/_qb.mean()-1):+.2f}%, p {_bp:.3f})")

HAR benchmark QLIKE = 0.1815   (802 origins)

--- (a) ML REPLACES HAR - direct, the same 10 domestic features ---
       model     QLIKE  vs HAR %      DM t      DM p   seed sd  mean |adj|
       Ridge    0.1819    0.2235   -1.8277    0.0680    0.0000      0.0051
         MLP    0.1825    0.5474   -0.2186    0.8270    0.0051      0.0766
  ElasticNet    0.1828    0.6969   -1.4246    0.1547    0.0000      0.0143
         XGB    0.1886    3.9066   -2.3615    0.0184    0.0006      0.0917
         GBM    0.1900    4.6786   -2.5286    0.0116    0.0000      0.0910
RandomForest    0.1968    8.4231   -2.1067    0.0355    0.0004      0.0953
BH-FDR 10%  BETTER than HAR: NONE

--- (b) ML on the HAR RESIDUAL - the same 10 domestic features ---
       model     QLIKE  vs HAR %      DM t      DM p   seed sd  mean |adj|
         MLP    0.1795   -1.0887    0.9980    0.3186    0.0048      0.0684
RandomForest    0.1798   -0.9338    1.5598    0.1192    0.0001      0.0392
         GBM    0.1799   -0.8929  

In [17]:
# Economic Evaluation
print("MINCER-ZARNOWITZ RECALIBRATION, applied symmetrically\n")
for c in ["HAR_bench"] + SAVG[:3] + ["COMB_equal_HAR_ML"]:
    raw = fc_boost[c].values
    rec, act = mz_recalibrate(a, raw); m = np.isfinite(rec) & act
    eff = mz_efficiency(a, raw)
    print(f"  {c:24} raw {qlike(a, raw).mean():.4f} -> recal "
          f"{qlike(a[m], rec[m]).mean():.4f}   MZ a={eff['a']:+.4f} b={eff['b']:.3f} "
          f"p={eff['p']:.4f}")

print("\n\nGIACOMINI-ROSSI FLUCTUATION TEST (seed-averaged)\n")
print(f"  {'model':22} {'sup|S|':>7} {'cv5':>7} {'flagged':>9} {'% windows ML ahead':>20}")
for c in SAVG + ["COMB_equal_HAR_ML"]:
    g = giacomini_rossi(q_har, qlike(a, fc_boost[c].values), frac=0.30)
    if g:
        print(f" {c:22} {g['sup']:7.2f} {g['cv5']:7.3f} {str(g['unstable']):>9}"  f" {100*g['frac_favouring_model']:19.1f}%")

print("\n\n VOLATILITY TIMING (Fleming-Kirby-Ostdiek 2001, 2003)\n")
r_next = feat["ret_d"].shift(-1).reindex(fc_boost.index).values
_rh = volatility_timing(r_next, fc_boost["HAR_bench"].values)
print(f"  {'model':22} {'Sharpe':>7} {'turnover':>9} {'fee vs HAR':>11} {'fee across seeds':>22}")
print(f"  {'HAR_bench':22} {_rh['sharpe']:7.3f} {_rh['turnover']:9.1f} {0.0:11.2f} {'-':>22}")
for _f in FAMS:
    _fees = [performance_fee(volatility_timing(r_next, fc_boost[f"BOOST_{_f}|s{s}"].values), _rh) for s in SEEDS]
    _r = volatility_timing(r_next, fc_boost[f"SEEDAVG_{_f}"].values)
    print(f"  {'SEEDAVG_'+_f:22} {_r['sharpe']:7.3f} {_r['turnover']:9.1f}"
          f" {performance_fee(_r, _rh):11.2f}   [{min(_fees):+8.1f}, {max(_fees):+7.1f}]")

print("\n\nPLACEBO - global features permuted across dates (must NOT improve)\n")
_rng = np.random.default_rng(RNG_SEED); _fs = feat.copy()
_perm = _rng.permutation(len(_fs))
for c in GB:
    _fs[c] = _fs[c].values[_perm]
_fsh = run_har_boost(_fs, GB, BLOCKS["DOMESTIC"], ["GBM", "XGB"], h=PRIMARY_HORIZON, seeds=(0,), verbose=False)                   
_ash = _fsh["actual"].values; _qsh = qlike(_ash, _fsh["HAR_bench"].values).mean()
for c in [c for c in _fsh.columns if c.startswith("BOOST_")]:
    _q = qlike(_ash, _fsh[c].values).mean()
    print(f"  {c:22} {_q:.4f} vs HAR {_qsh:.4f}   "
          f"{'*** LEAK ***' if _q < _qsh - 1e-4 else 'ok - no spurious gain'}")

MINCER-ZARNOWITZ RECALIBRATION, applied symmetrically

  HAR_bench                raw 0.1815 -> recal 0.2061   MZ a=-0.3427 b=0.987 p=0.0000
  SEEDAVG_Ridge            raw 0.1819 -> recal 0.2078   MZ a=-0.3733 b=0.983 p=0.0000
  SEEDAVG_ElasticNet       raw 0.1814 -> recal 0.2061   MZ a=-0.3282 b=0.988 p=0.0000
  SEEDAVG_RandomForest     raw 0.1822 -> recal 0.2062   MZ a=-0.4498 b=0.974 p=0.0000
  COMB_equal_HAR_ML        raw 0.1827 -> recal 0.2073   MZ a=-0.4323 b=0.976 p=0.0000


GIACOMINI-ROSSI FLUCTUATION TEST (seed-averaged)

  model                   sup|S|     cv5   flagged   % windows ML ahead
 SEEDAVG_Ridge             2.24   3.012     False                44.4%
 SEEDAVG_ElasticNet        2.27   3.012     False                49.9%
 SEEDAVG_RandomForest      2.46   3.012     False                36.8%
 SEEDAVG_GBM               2.93   3.012     False                17.2%
 SEEDAVG_XGB               3.25   3.012      True                13.0%
 SEEDAVG_MLP               4.10   3.

In [18]:
# SHAP Permutation against its permutation null
SHAP_MODEL = "XGB"                       
ALLF = BLOCKS["DOMESTIC"] + BLOCKS["DOM_EXT"] + BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"]
_us   = feat[feat["logRV_target"].notna()]
_n0   = int(len(_us) * (1 - OOS_FRACTION))
_tr, _ev = _us.index[:_n0], _us.index[_n0:]
_X    = feat[ALLF].astype(float)
Xtr, ytr = _X.loc[_tr].values, _us["logRV_target"].loc[_tr].values
Xev      = _X.loc[_ev].values
_hp = GRIDS[SHAP_MODEL][0]
print(f"{len(ALLF)} features | fit on {len(_tr)} rows ({_tr[0].date()}->{_tr[-1].date()})"
      f" | explain {len(_ev)} OOS rows ({_ev[0].date()}->{_ev[-1].date()})\n")

# --- Null A: Altmann target permutation -------------------------------
shap_tab, SV, _ = shap_with_null(SHAP_MODEL, Xtr, ytr, Xev, ALLF, hp=_hp, B=SHAP_NULL_B, verbose=False)
print("NULL A - block-permuted TARGET (Altmann et al. 2010)\n")
print(shap_tab.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
_sigA = benjamini_hochberg(shap_tab["perm p"].values, list(shap_tab["feature"]), alpha=0.10)
print(f"\nSurvive BH-FDR 10%: {sorted(_sigA) if _sigA else 'NONE'}")

# --- Null B: conditional randomisation on the global block ------------
_thr  = feat["RV"].expanding(min_periods=100).quantile(2/3).shift(1).reindex(_ev)
_turb = (feat["RV"].reindex(_ev) >= _thr).values
GLOB = BLOCKS["GLOBAL_ALL"] + BLOCKS["INTERACT"]
cond_tab, _, state_tab = shap_conditional_null(
    SHAP_MODEL, Xtr, ytr, Xev, ALLF, GLOB, hp=_hp, B=SHAP_NULL_B, verbose=False,
    state_mask=_turb, state_blocks=BLOCKS)
print("\n\nNULL B - global columns block-permuted, target and domestic intact\n"
      "(conditional randomisation test, Candes et al. 2018)\n")
print(cond_tab.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
_sigB = benjamini_hochberg(cond_tab["cond p"].values, list(cond_tab["feature"]), alpha=0.10)
print(f"\nSurvive BH-FDR 10%: {sorted(_sigB) if _sigB else 'NONE'}")
LED_shap_sigB  = sorted(_sigB) if _sigB else []
LED_dom_share  = 100 * shap_tab[shap_tab["feature"].isin(BLOCKS["DOMESTIC"])]["mean|SHAP|"].sum() / shap_tab["mean|SHAP|"].sum()
LED_dom_null   = 100 * shap_tab[shap_tab["feature"].isin(BLOCKS["DOMESTIC"])]["null mean"].sum() / shap_tab["null mean"].sum()
LED_state_tab  = state_tab.copy()

print("\n\nATTRIBUTION BY BLOCK\n")
_rows = []
for _b in ("DOMESTIC", "DOM_EXT", "GLOBAL_IV", "GLOBAL_FX", "GLOBAL_CMD", "GLOBAL_EQ", "INTERACT"):
    _c = [c for c in BLOCKS.get(_b, []) if c in ALLF]
    if not _c:
        continue
    _s = shap_tab[shap_tab["feature"].isin(_c)]
    _rows.append({"block": _b, "k": len(_c), "sum mean|SHAP|": _s["mean|SHAP|"].sum(),  "share %": 100*_s["mean|SHAP|"].sum()/shap_tab["mean|SHAP|"].sum(),  "null-A share %": 100*_s["null mean"].sum()/shap_tab["null mean"].sum(), "min p (A)": _s["perm p"].min()})
print(pd.DataFrame(_rows).to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

32 features | fit on 1869 rows (2015-06-10->2023-02-10) | explain 802 OOS rows (2023-02-13->2026-05-15)

NULL A - block-permuted TARGET (Altmann et al. 2010)

      feature  mean|SHAP|  null mean  null q95     ratio    perm p
      logRV_1      0.2415     0.0110    0.0275   22.0364    0.0050
      logRV_5      0.1307     0.0345    0.0797    3.7879    0.0050
      r_neg_1      0.0485     0.0046    0.0146   10.5454    0.0050
      r_pos_1      0.0448     0.0037    0.0112   12.2354    0.0050
      r_neg_5      0.0379     0.0207    0.0441    1.8265    0.0995
      logIV_1      0.0373     0.0573    0.1251    0.6511    0.6517
      r_pos_5      0.0256     0.0159    0.0329    1.6127    0.1244
     r_pos_22      0.0235     0.0363    0.0792    0.6460    0.7015
  logIV_slope      0.0209     0.0306    0.0670    0.6823    0.6219
    logVoV_22      0.0161     0.0406    0.0733    0.3970    0.9104
     r_neg_22      0.0158     0.0462    0.0982    0.3415    0.9154
     logRV_22      0.0157     0.0763 

In [19]:
print("SHAP DEPENDENCE - how much of each feature's attribution is a straight line?\n")
_top = list(shap_tab["feature"].head(8))
print(shap_linearity(SV, Xev, ALLF, _top).to_string(index=False,float_format=lambda x: f"{x:9.4f}"))
print("\n\nINTERACTION SHARE (TreeSHAP interaction values, exact)\n")
_m = make_model(SHAP_MODEL, seed=RNG_SEED, **_hp); _m.fit(Xtr, ytr)
_ev_sub = Xev[:SHAP_INTER_ROWS]
_ii = shap_interaction_share(_m, _ev_sub, SHAP_MODEL, ALLF)
print(f"  observed off-diagonal share = {100*_ii['interaction_share']:.2f}%")
print("  largest pairwise terms:")
for _a, _b2, _v in _ii["top_pairs"][:6]:
    print(f"    {_a:16} x {_b2:16} {_v:.5f}")
_rng = np.random.default_rng(RNG_SEED); _ns = []
for _b in range(SHAP_INTER_B):
    _yb = _circular_block_permute(ytr, SHAP_BLOCK, _rng)
    _mb = make_model(SHAP_MODEL, seed=RNG_SEED+1+_b, **_hp); _mb.fit(Xtr, _yb)
    _ns.append(shap_interaction_share(_mb, _ev_sub, SHAP_MODEL, ALLF)["interaction_share"])
_ns = np.asarray(_ns)
print(f"\n  null-A share: mean {100*_ns.mean():.2f}%  q95 {100*np.quantile(_ns,0.95):.2f}%"
      f"   permutation p = {(1+(_ns>=_ii['interaction_share']).sum())/(len(_ns)+1):.4f}")
LED_inter_share = float(_ii["interaction_share"])
LED_inter_null  = float(np.mean(_ns))

print("\n\nATTRIBUTION BY EX-ANTE VOLATILITY STATE\n")
_st, _tb = shap_by_state(SV, _ev, feat, BLOCKS, ALLF)
print(f"  turbulent origins: {int(_tb.sum())} / {len(_tb)}\n")
print(_st.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
print("\n\nTHE SAME RATIOS AGAINST THEIR CONDITIONAL NULL\n")
print(state_tab.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

SHAP DEPENDENCE - how much of each feature's attribution is a straight line?

 feature  R2 linear  R2 cubic  curvature     slope
 logRV_1     0.9205    0.9678     0.0472    0.2785
 logRV_5     0.7269    0.9201     0.1932    0.1554
 r_neg_1     0.7528    0.9239     0.1711   -5.2963
 r_pos_1     0.7022    0.8524     0.1502    5.6640
 r_neg_5     0.5165    0.8182     0.3017   -6.2117
 logIV_1     0.8138    0.8536     0.0398    0.0642
 r_pos_5     0.2426    0.6272     0.3846    3.2581
r_pos_22     0.6549    0.7670     0.1122    8.6892


INTERACTION SHARE (TreeSHAP interaction values, exact)

  observed off-diagonal share = 22.86%
  largest pairwise terms:
    logRV_1          x logRV_5          0.00843
    logIV_1          x r_pos_5          0.00384
    logRV_5          x logRQ_1          0.00369
    r_pos_1          x inr_vol_22       0.00278
    logIV_1          x r_pos_22         0.00266
    r_pos_1          x logVoV_22        0.00254

  null-A share: mean 37.26%  q95 43.58%   permutati

In [20]:
# Robustness check for SHAP Finding
_thr2  = feat["RV"].expanding(min_periods=100).quantile(2/3).shift(1)
feat_T = feat.assign(TURB=(feat["RV"] >= _thr2).astype(float))
_D  = BLOCKS["DOMESTIC"]; _y = feat_T["logRV_target"]
_ok = _y.notna() & feat_T["TURB"].notna()
_r0 = sm.OLS(_y[_ok], sm.add_constant(feat_T[_D + ["TURB"]])[_ok]).fit(
    cov_type="HAC", cov_kwds={"maxlags": 10})

print("1. LINEAR REGIME-INTERACTION TEST")
print(f"n = {int(_ok.sum()):,}   turbulent share {feat_T.loc[_ok,'TURB'].mean():.3f}"
      f"base R2 = {_r0.rsquared:.4f}\n")
_rows = []
for _nm, _cols in (("GLOBAL_IV", BLOCKS["GLOBAL_IV"]), ("GLOBAL_FX", BLOCKS["GLOBAL_FX"]), ("GLOBAL_CMD", BLOCKS["GLOBAL_CMD"]), ("GLOBAL_EQ", BLOCKS["GLOBAL_EQ"]), ("ALL GLOBAL", BLOCKS["GLOBAL_ALL"])):
    _g = feat_T.copy(); _ix = []
    for _c in _cols:
        _g[f"T_{_c}"] = _g[_c] * _g["TURB"]; _ix.append(f"T_{_c}")
    _X1 = sm.add_constant(_g[_D + ["TURB"] + _cols + _ix])
    _r1 = sm.OLS(_y[_ok], _X1[_ok]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
    def _wald(cs):
        R = np.zeros((len(cs), _X1.shape[1]))
        for _j, _c in enumerate(cs):
            R[_j, list(_X1.columns).index(_c)] = 1.0
        w = _r1.f_test(R); return float(w.fvalue), float(w.pvalue)
    _fi, _pi = _wald(_ix); _fl, _pl = _wald(_cols)
    _rows.append({"block": _nm, "k": len(_cols),"F(interaction)": _fi, "p(interaction)": _pi,"F(calm level)": _fl, "p(calm level)": _pl,"dR2 x100": 100*(_r1.rsquared - _r0.rsquared)})
_rt = pd.DataFrame(_rows)
print(_rt.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))
_sg = benjamini_hochberg(_rt["p(interaction)"].values, list(_rt["block"]), alpha=0.10)
LED_regime_sig = sorted(_sg) if _sg else []
print(f"\n   interaction terms surviving BH-FDR 10%: {sorted(_sg) if _sg else 'NONE'}")
print("\n\n2. SPLIT-SAMPLE TEST - the block fitted inside each state, no pooling\n")
for _nm, _cols in (("GLOBAL_IV", BLOCKS["GLOBAL_IV"]), ("GLOBAL_EQ", BLOCKS["GLOBAL_EQ"]),  ("ALL GLOBAL", BLOCKS["GLOBAL_ALL"])):
    for _lab, _m in (("calm", feat_T["TURB"] == 0), ("turbulent", feat_T["TURB"] == 1)):
        _mk = _ok & _m
        _ra = sm.OLS(_y[_mk], sm.add_constant(feat_T[_D])[_mk]).fit(
            cov_type="HAC", cov_kwds={"maxlags": 10})
        _X1 = sm.add_constant(feat_T[_D + _cols])
        _rb = sm.OLS(_y[_mk], _X1[_mk]).fit(cov_type="HAC", cov_kwds={"maxlags": 10})
        R = np.zeros((len(_cols), _X1.shape[1]))
        for _j, _c in enumerate(_cols):
            R[_j, list(_X1.columns).index(_c)] = 1.0
        _w = _rb.f_test(R)
        print(f"   {_nm:11} in {_lab:10} n={int(_mk.sum()):5d}  dR2x100 "
              f"{100*(_rb.rsquared-_ra.rsquared):7.4f}  F {float(_w.fvalue):6.3f}"
              f"  p {float(_w.pvalue):.4f}")

print("\n\n3. OUT-OF-SAMPLE - the HAR-boost comparison restricted to each state\n")
_tb = (feat_T["TURB"].reindex(fc_boost.index) == 1).values
_CAND = SAVG + ["COMB_equal_HAR_ML", "COMB_invloss"]
regime_oos = {}
for _lab, _m in (("calm", ~_tb), ("turbulent", _tb), ("all", np.ones_like(_tb))):
    _qh = qlike(a[_m], fc_boost["HAR_bench"].values[_m])
    _best, _bq, _bp = None, np.inf, np.nan
    for _c in _CAND:
        _q = qlike(a[_m], fc_boost[_c].values[_m])
        if _q.mean() < _bq:
            _best, _bq = _c, _q.mean(); _bp = dm(_qh, _q)[1]
    regime_oos[_lab] = (_best, 100*(_bq/_qh.mean()-1), _bp)
    print(f"   {_lab:10} n={int(_m.sum()):4d}   HAR {_qh.mean():.4f}   best {_best:22}"
          f" {_bq:.4f}  ({100*(_bq/_qh.mean()-1):+.2f}%, DM p {_bp:.3f})")

1. LINEAR REGIME-INTERACTION TEST
n = 2,671   turbulent share 0.279base R2 = 0.5074

     block  k  F(interaction)  p(interaction)  F(calm level)  p(calm level)  dR2 x100
 GLOBAL_IV  3          1.7949          0.1460         2.2971         0.0757    0.2816
 GLOBAL_FX  3          1.8058          0.1439         0.5516         0.6471    0.1349
GLOBAL_CMD  3          4.6637          0.0030         2.7228         0.0429    0.3744
 GLOBAL_EQ  3          1.3139          0.2681         1.7652         0.1517    0.1554
ALL GLOBAL 12          2.2050          0.0095         1.2301         0.2555    0.7638

   interaction terms surviving BH-FDR 10%: ['ALL GLOBAL', 'GLOBAL_CMD']


2. SPLIT-SAMPLE TEST - the block fitted inside each state, no pooling

   GLOBAL_IV   in calm       n= 1927  dR2x100  0.3059  F  3.319  p 0.0191
   GLOBAL_IV   in turbulent  n=  744  dR2x100  0.5422  F  2.987  p 0.0305
   GLOBAL_EQ   in calm       n= 1927  dR2x100  0.2547  F  3.027  p 0.0285
   GLOBAL_EQ   in turbulent  n=

In [21]:
# Comparison And Summary
def _verdict(p, better, alpha=0.05):
    if not np.isfinite(p):
        return "not testable"
    d = "BETTER" if better else "WORSE"
    return f"{d}, p={p:.3f} -> {'SIGNIFICANT' if p < alpha else 'not significant'}"

L = []
def _add(sec, comparison, metric, result):
    L.append({"#": len(L)+1, "§": sec, "comparison": comparison,"metric": metric, "one-line result": result})

for _, _row in rq1_tab.iterrows():
    _add("7", f"{_row['block']} added to domestic benchmark", "HAC Wald",
         f"dR2={_row['dR2 x100']:.3f}%, p={_row['p']:.3f} -> "
         f"{'significant' if _row['p'] < 0.05 else 'NOT significant'}")
_add("7", "DOM_EXT added to domestic benchmark (POWER CHECK)", "HAC Wald",
     f"dR2={domext_dr2:.3f}%, p={domext_p:.4f} -> "
     f"{'significant: the design CAN detect this effect size' if domext_p < 0.05 else 'not significant'}")
_add("7", f"{solo_n} single global regressors, one at a time", "BH-FDR 10%",
     f"{solo_sig} survive" if solo_sig else f"NONE of {solo_n} survive")

_bt = bench_table(fc_bench)
for _, _r in _bt.iterrows():
    _add("8", f"{_r['benchmark']} vs Part I's selected model", "QLIKE / DM",
         "reference" if not np.isfinite(_r["DM p"]) else
         f"{_r['vs selected %']:+.2f}%, p={_r['DM p']:.4f}")

for _, _r in mx_tab.iterrows():
    _add("9", f"best ML on {_r['benchmark']} | features={_r['ML features']}",
         "QLIKE / DM + BH",
         f"{_r['best ML']} {_r['best vs bench %']:+.2f}%, p={_r['best DM p']:.3f}"
         f" -> {_r['n better @FDR10']} beat the benchmark at FDR 10%")

for _f in FAMS:
    _q = qlike(a, fc_boost[f"SEEDAVG_{_f}"].values)
    _t, _p = dm(q_har, _q)
    _add("10", f"{_f} (seed-averaged) vs log-HAR-RV-IV-LFULL", "QLIKE / DM",
         _verdict(_p, _q.mean() < q_har.mean()))
for _c in ("COMB_equal_ML", "COMB_equal_HAR_ML", "COMB_invloss"):
    _q = qlike(a, fc_boost[_c].values); _t, _p = dm(q_har, _q)
    _add("10", f"{_c} vs HAR", "QLIKE / DM", _verdict(_p, _q.mean() < q_har.mean()))
_add("10", "all candidates jointly", "Model Confidence Set",
     f"{LED_mcs[0]} of {LED_mcs[1]} retained; HAR {'IS' if LED_mcs[2] else 'is NOT'} "
     f"in the superior set")

_add("11", "mean |adjustment| vs QLIKE penalty across 8 families", "Spearman",
     f"rho={LED_rho:+.3f} -> loss rises monotonically with deviation from HAR")

for _lbl, _tt in (("ML replaces HAR, same features", tab_a),  ("ML on HAR residual, same features", tab_b), ("ML on HAR residual, DOM_EXT (positive control)", tab_c)):
    _b = _tt.iloc[0]
    _add("12", _lbl, "QLIKE / DM",
         f"best={_b['model']} {_b['vs HAR %']:+.2f}%, p={_b['DM p']:.3f}")
_add("12", "OLS with DOM_EXT vs HAR (interprets the control)", "QLIKE / DM",
     f"{LED_ols_domext[0]:+.2f}%, p={LED_ols_domext[1]:.3f} -> "
     f"a LINEAR model fails too, so the constraint is estimation noise")

_e = mz_efficiency(a, fc_boost["HAR_bench"].values)
_add("13", "HAR forecast unbiased and efficient?", "Mincer-Zarnowitz",
     f"a={_e['a']:+.3f}, b={_e['b']:.3f}, joint p={_e['p']:.4f}")
_g = giacomini_rossi(q_har, qlike(a, fc_boost["SEEDAVG_XGB"].values), frac=0.30)
_add("13", "is relative performance stable over time?", "Giacomini-Rossi",
     f"XGB sup|S|={_g['sup']:.2f} but ML ahead in {100*_g['frac_favouring_model']:.1f}% "
     f"of windows -> stably behind, not unstable")
_rnext = feat["ret_d"].shift(-1).reindex(fc_boost.index).values      # recomputed locally
_rhar  = volatility_timing(_rnext, fc_boost["HAR_bench"].values)
_fees = [performance_fee(volatility_timing(_rnext, fc_boost[f'BOOST_MLP|s{s}'].values), _rhar)
         for s in SEEDS]
_add("13", "economic value to a volatility-timing investor", "FKO fee",
     f"MLP fee spans [{min(_fees):+.0f}, {max(_fees):+.0f}] bps across seeds "
     f"-> not separable from simulation noise")
_add("13", "placebo: global features permuted across dates", "QLIKE",
     "no model improves on HAR -> the machinery does not manufacture gains")

_add("14", "where does an unrestricted learner put its attribution?", "mean |SHAP|",
     f"{LED_dom_share:.1f}% on the 10 domestic columns (null share {LED_dom_null:.1f}%)")
_add("14", "any global feature above its conditional null?", "Null B + BH-FDR",
     f"{LED_shap_sigB} survive" if LED_shap_sigB else "NONE survive")
_add("14", "is the fitted function additive (HAR-shaped)?", "TreeSHAP interaction",
     f"{100*LED_inter_share:.1f}% off-diagonal vs {100*LED_inter_null:.1f}% on permuted "
     f"targets -> {'MORE' if LED_inter_share < LED_inter_null else 'LESS'} additive than noise")

_giv = LED_state_tab[LED_state_tab["block"] == "GLOBAL_IV"].iloc[0]
_add("15", "GLOBAL_IV attribution, turbulent vs calm", "SHAP + conditional null",
     f"ratio {_giv['turb/calm']:.2f} vs null {_giv['null mean']:.2f}, p={_giv['perm p']:.3f}")
_add("15", "global x state interaction (linear replication)", "HAC Wald + BH",
     f"{LED_regime_sig if LED_regime_sig else 'NONE'} survive FDR 10%; none significant in LEVELS")
_add("15", "out-of-sample, TURBULENT origins only", "QLIKE / DM",
     f"best ML {regime_oos['turbulent'][1]:+.2f}% vs HAR, p={regime_oos['turbulent'][2]:.3f}"
     f" -> {'significant' if regime_oos['turbulent'][2] < 0.05 else 'a direction, not a result'}")
_add("15", "out-of-sample, CALM origins only", "QLIKE / DM",
     f"best ML {regime_oos['calm'][1]:+.2f}% vs HAR, p={regime_oos['calm'][2]:.3f}")

ledger = pd.DataFrame(L)
print(f"COMPARISON LEDGER - {len(ledger)} formal comparisons, in the order they are made\n")
with pd.option_context("display.max_colwidth", 78, "display.width", 200):
    print(ledger.to_string(index=False))

COMPARISON LEDGER - 47 formal comparisons, in the order they are made

 #  §                                              comparison                  metric                                                                    one-line result
 1  7             GLOBAL_EQ (S&P) added to domestic benchmark                HAC Wald                                             dR2=0.095%, p=0.126 -> NOT significant
 2  7             GLOBAL_IV (VIX) added to domestic benchmark                HAC Wald                                                 dR2=0.178%, p=0.029 -> significant
 3  7                   GLOBAL_FX added to domestic benchmark                HAC Wald                                             dR2=0.034%, p=0.637 -> NOT significant
 4  7                  GLOBAL_CMD added to domestic benchmark                HAC Wald                                             dR2=0.116%, p=0.081 -> NOT significant
 5  7                  ALL GLOBAL added to domestic benchmark                HAC Wal